# 2026 Senate Forecast — Marks_Final

**Marks_Final** is the team's final-model candidate. It is **Pray_For_The_Floor** (Model 1's Bayesian backbone on live VoteHub + Kalshi data, with DadBod's controls), with every assumption that could be checked against history **calibrated on past Senate races**, and with the structural problems found in the Oct 29 model review fixed.

| Change from Pray_For_The_Floor | Why | Where |
|---|---|---|
| **Calibration from history** (`calibration.json`, written by `Final Model/calibration.ipynb`) | The four settings that moved P(51+) the most (prior SD, poll error, correlation, tails) were guesses; now they're measured on every competitive Senate race 1998–2022 | after CONFIG |
| **Candidate effects in the prior** (`candidate_history.csv`) | The lean-based prior ignored who's running: Maine's prior was D+13.5 against Collins, Alaska and Nebraska ignored Peltola's and Osborn's past over-performance | Step 6 |
| **Errors simulated as national + regional + lean + state pieces, in points** | Pray's one shared fat-tail draw produced 35- and 69-seat outcomes and made a national miss bigger in unpolled safe states | Step 7 |
| **Kalshi as a real input**: fixed conversion SD, longshot correction, only informative prices, weight ~2× Pray's | Pray's conversion was circular, pulled safe races to the middle, and at 5–7% weight Kalshi changed nothing | Step 6 |
| **National mood**: generic ballot weighted 0.75, stale VoteHub polls count as fewer "fresh" polls | VoteHub's national feed is weeks old; the generic ballot is already a margin | Step 5 |
| **In-cycle house effects**: each pollster's lean measured from this year's polls (vs other pollsters in the same states), shrunk toward its scorecard lean | Tested on 1998–2022 Senate races: polling-average error 4.72 → 4.43 pts, better Brier in 10 of 15 cycles | Step 4 |
| **RV shift measured** from 2026 LV/RV survey pairs; LV version preferred | Pray assumed RV polls lean D by 0.5; 2026 pairs show the opposite | Steps 2 and 4 |
| **Kalshi requests retried and paged** | Pray's search failed silently under rate limits, so different races fell back to the hand table each run | Step 1 |
| **Deciding-seat table** (from Rylyn's model) | Shows which races actually decide control | Step 9 |

Everything else (VoteHub polls, scorecard grades, random grades for ungraded pollsters redrawn every simulation, flooding penalty, version splitting, scorecard house effects, correlated source errors, half-life recency, standard export, Model Testing) is Pray's code, unchanged.

### What it produces
1. **P(Democrats hold at least 51 seats)**: with a Republican vice president, 51 is the number Democrats need for control
2. **The predicted number of Democratic seats**: the average over all simulated elections, counting independents who caucus with Democrats
3. **Every race's win probability and projected margin**

### The model in one line
For each of the 35 races, three estimates of the margin are combined by **precision** (1 ÷ uncertainty²), so the most certain source counts most:

$$\text{margin} = \frac{\pi_F \cdot \text{prior} + \pi_P \cdot \text{polls} + \pi_K \cdot \text{Kalshi}}{\pi_F + \pi_P + \pi_K}, \qquad \pi = \frac{1}{\text{SD}^2}$$

| Symbol | What it is | Set by |
|---|---|---|
| prior | Fundamentals: 2 × partisan lean + national environment ± incumbency + candidate effect (Steps 5–6) | `PVI_TO_MARGIN`, `ENV_*`, `INCUMBENCY_*`, `CANDIDATE_*` |
| polls | The race's weighted polling average (Steps 2–4) | poll-weight settings |
| Kalshi | The market's win probability turned into a margin (Step 6) | `KALSHI_*` |
| π_F | Precision of the prior: 1 ÷ `PRIOR_SD`² | `PRIOR_SD` (calibrated) |
| π_P | Precision of the polls: shrinks with fewer or noisier polls; never more than 1 ÷ `SYSTEMATIC_POLL_ERROR_SD`² | `SYSTEMATIC_POLL_ERROR_SD` (calibrated) |
| π_K | Precision of Kalshi: 1 ÷ `KALSHI_SD`² | `KALSHI_SD` |

### How to use
1. Run the **install** cell once per computer.
2. (Optional) re-run `Final Model/calibration.ipynb` to refresh `calibration.json`.
3. Change anything in **CONFIG** (each line says what it does; `MARKS` marks what's new). The cell after it shows the formula with your numbers filled in.
4. *Run All.* Results land in `Output/` in this folder. Then open **`Model Testing.ipynb`**.

All margins are **Democrat (or a credible independent) minus Republican**, in percentage points. D+3 means the Democrat leads by 3.


## Glossary (plain English)

| Term | Meaning |
|---|---|
| **Margin** | Democratic % minus Republican %. D+4 = Democrat ahead by 4 points; R+4 = Republican ahead by 4. |
| **Poll** | A survey of a few hundred to a few thousand people, used to estimate how the whole state will vote. |
| **Sample size (n)** | How many people a poll interviewed. Bigger samples have less random error. |
| **LV / RV / A** | Who was polled: *likely voters*, *registered voters*, or *all adults*. LV is closest to who actually votes. |
| **Pollster grade** | VoteHub's report card (A+ to D) for how accurate a polling firm has been. Many firms have no grade. |
| **House effect** | A pollster's typical lean: how far its results usually land from the final result, and in which direction. |
| **Partisan poll** | A poll paid for by a party or campaign. These usually flatter their sponsor. |
| **Flooding** | One firm releasing many polls of the same race, which would let it dominate the average. |
| **Generic ballot** | The national question "would you vote for the Democrat or the Republican for Congress?" A read on the national mood. |
| **Net approval** | % who approve of the president minus % who disapprove. |
| **Partisan lean (PVI)** | How much more Republican or Democratic a state votes than the country, in vote-share points. Lean −15 = 15 points more Republican. |
| **Prior / fundamentals** | What we'd expect in a race before looking at its polls: partisan lean, the national mood, incumbency. |
| **Precision** | 1 ÷ (uncertainty²). A source that's twice as certain counts four times as much. This is how a Bayesian update weighs sources. |
| **Posterior** | The final estimate after combining the prior with the polls (and Kalshi). |
| **Kalshi** | A regulated prediction market. A price of 0.64 means the market gives about a 64% chance. |
| **Simulation** | One made-up election drawn from the model's ranges. Running 10,000 of them shows how often each outcome happens. |
| **Correlation** | When states tend to miss in the same direction (a polling error in Ohio usually shows up in Michigan too). |
| **Fat tails** | Big surprises (like 2016) happen more often than a bell curve suggests. The model uses a Student-t curve to allow for them. |
| **Backtest** | Re-running the model as if it were an earlier date, using only what was known then (`AS_OF_DATE`). |
| **Candidate effect** | How much better or worse a nominee ran than a generic candidate of their party in their last statewide race. |
| **Calibration** | Setting the model's uncertainty sizes from how far past forecasts actually missed. |
| **Deciding seat** | In one simulated election, the race that gives a party its majority. |

## Setup: install packages
Run once per computer (safe to re-run; packages you already have are skipped). Installs everything this notebook and `Model Testing.ipynb` need that doesn't ship with Python. Restart the kernel if anything was newly installed.

In [ ]:
%pip install --quiet numpy pandas scipy matplotlib jinja2 nbformat

In [ ]:
import copy  # deep-copy CONFIG for sweep runs
import json  # parse API responses
import os
import re  # normalize names
import time  # pauses between Kalshi requests
import unicodedata  # strip accents from candidate names
import urllib.error
import urllib.request  # stdlib HTTP (no extra install needed)
from pathlib import Path  # OS-independent file paths

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import norm
from IPython.display import Markdown, display

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)

## CONFIG — every assumption lives here

Same layout as Model 1, grouped by model stage. **New or changed** settings are marked `NEW` / `CHANGED` in their notes. Change a value, then *Run All*. The Model Testing notebook's sweep changes one key at a time, so keys are referenced by name there.

In [ ]:
# Repo root = first folder at or above the working directory that contains Data/.
REPO_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "Data").is_dir())
MODEL_DIR = REPO_ROOT / "Models" / "Marks_Final"  # this notebook's folder

CONFIG = {
    # ---------------- Files & run settings ----------------
    "DATA_DIR": REPO_ROOT / "Data",  # shared data folder (read-only)
    "OUT_DIR": MODEL_DIR / "Output",  # results written here (created if missing)
    "CACHE_DIR": MODEL_DIR / "Data_Cache",  # NEW: last good download of every source, used if a website is down
    "LEANS_FILE": "leans_and_regions.csv",  # partisan lean (PVI) + Census region per state
    "SCORECARD_FILE": "votehub_pollster_scorecards.csv",  # VoteHub grade, % error, house effect per pollster
    "CANDIDATE_FILE": MODEL_DIR / "candidate_history.csv",  # MARKS: each nominee's last statewide result (Step 6)
    "USE_CALIBRATION": True,  # MARKS: True = values in calibration.json (written by Final Model/calibration.ipynb)
                              #   replace the CONFIG values below. False = use CONFIG exactly as written
    "CALIBRATION_FILE": MODEL_DIR / "calibration.json",
    "ELECTION_DAY": "2026-11-03",
    "AS_OF_DATE": None,  # None = today. "2026-10-15" = backtest: only polls RELEASED by then, that day's Kalshi race
                         #   prices, and polls only for the national numbers (Kalshi's national averages are skipped)
    "N_SIMS": 10_000,  # simulated elections. 10,000 gives about +/-1 pt (95%) on P(51+)
    "SEED": 2026,  # same seed + same data = same numbers every run

    # ---------------- Senate map & control rules ----------------
    "SEATS_UP": {  # party currently holding each seat on the 2026 ballot (33 Class II + FL/OH specials)
        "Alabama": "R", "Alaska": "R", "Arkansas": "R", "Florida": "R", "Idaho": "R",
        "Iowa": "R", "Kansas": "R", "Kentucky": "R", "Louisiana": "R", "Maine": "R",
        "Mississippi": "R", "Montana": "R", "Nebraska": "R", "North Carolina": "R",
        "Ohio": "R", "Oklahoma": "R", "South Carolina": "R", "South Dakota": "R",
        "Tennessee": "R", "Texas": "R", "West Virginia": "R", "Wyoming": "R",
        "Colorado": "D", "Delaware": "D", "Georgia": "D", "Illinois": "D",
        "Massachusetts": "D", "Michigan": "D", "Minnesota": "D", "New Hampshire": "D",
        "New Jersey": "D", "New Mexico": "D", "Oregon": "D", "Rhode Island": "D",
        "Virginia": "D",
    },
    "HOLDOVER_D": 34,  # D + caucusing independents NOT on the ballot (47 - 13 up)
    "HOLDOVER_R": 31,  # R not on the ballot (53 - 22 up)
    "VP_PARTY": "R",  # VP breaks 50-50 ties -> with "R", Democrats need 51
    "D_SEATS_FOR_CONTROL": None,  # None = derive from VP_PARTY (51 or 50). 52 = the "Fetterman switches" view
    "HOLDOVER_DEFECTION_PROB": {"Fetterman (PA)": 0.0},  # P(holdover D-caucus senator organizes with R), drawn per simulation
    "INDEPENDENT_CAUCUS_PROB_D": {"Nebraska": 0.8},  # P(independent caucuses with D if they win): Adler's 80% for Osborn
    # Incumbent status for the prior's incumbency bonus: "running", "appointed", "open". Hand-maintained; re-check before the final run.
    "INCUMBENT_STATUS": {
        "Illinois": "open", "Michigan": "open", "Minnesota": "open", "New Hampshire": "open",
        "Alabama": "open", "Iowa": "open", "Kentucky": "open", "Montana": "open",
        "North Carolina": "open", "Wyoming": "open", "Oklahoma": "open", "Texas": "open", "Louisiana": "open",
        "Florida": "appointed", "Ohio": "appointed", "South Carolina": "appointed",
    },
    "DEFAULT_INCUMBENT_STATUS": "running",  # everyone else

    # ---------------- Data cleaning ----------------
    "POLLSTER_NAME_FIXES": {},  # manual spelling fixes, e.g. {"Morning consult": "Morning Consult"}; case/punctuation
                                #   differences are merged automatically
    "MIN_SAMPLE_SIZE": 0,  # drop polls smaller than this (0 = keep all)
    "MISSING_SAMPLE_SIZE": 600,  # NEW: size assumed when VoteHub doesn't report one
    "MAX_POLL_AGE_DAYS": None,  # drop polls older than this (None = keep all; recency weighting handles age)
    "DROP_EXACT_DUPLICATES": True,  # same pollster/state/date/numbers listed twice -> keep one
    "NAT_ZERO_SAMPLE_IMPUTE": 1000,  # national polls reporting n = 0 get this size instead of zero weight

    # ---------------- Multi-version polls (same pollster/state/end date released several ways) ----------------
    # "split" (DEFAULT): keep every version, each gets 1/(# versions) so the survey counts once in total
    # "keep_all": each version counts as a separate poll | "prefer_h2h" / "prefer_full_field": keep only that version when one exists
    "POLL_VERSION_MODE": "split",
    "PREFER_LV_VERSION": True,  # MARKS: survey released for likely AND registered voters -> keep only the LV version
    "DROP_GENERIC_IF_CANDIDATE_VERSION": True,  # survey with both a generic (party) and a named-candidate version -> drop the generic

    # ---------------- Undecided / other voters ----------------
    # "Other" = 100 - Dem% - Rep%. Part of it is truly undecided; the rest sticks with third parties.
    # CHANGED: "full field" is now detected automatically (VoteHub lists more than two candidates)
    "OTHER_TO_UNDECIDED_H2H": 1.0,  # head-to-head: all non-D/R is undecided
    "OTHER_TO_UNDECIDED_FULL_FIELD": 0.5,  # full field: half of "other" is real third-party support
    # How undecideds break: "ignore" (raw D-R), "proportional" (like decided voters), "fixed_share" (D gets UNDECIDED_D_SHARE),
    # "against_incumbent" (challenger gets UNDECIDED_CHALLENGER_SHARE; open seats split evenly)
    "UNDECIDED_MODE": "proportional",
    "UNDECIDED_D_SHARE": 0.5,
    "UNDECIDED_CHALLENGER_SHARE": 0.55,
    "UNDECIDED_SPLIT_SD": 0.15,  # uncertainty in the share of undecideds going D; adds 2*U*SD pts of margin SD per poll

    # ---------------- Poll weights ----------------
    "RECENCY_MODE": "half_life",  # NEW: "half_life" = Model 1 (every poll counts, halving every RECENCY_HALF_LIFE_DAYS);
                                  #   "window" = Ozempic/DadBod (only the last WINDOW_DAYS, weight falling in a straight line)
    "RECENCY_HALF_LIFE_DAYS": 21,  # "half_life" mode: after ~3 weeks a poll is half as informative
    "WINDOW_DAYS": 28,  # NEW, "window" mode: polls older than this are dropped
    "WINDOW_OLDEST_WEIGHT": 0.25,  # NEW, "window" mode: weight of a poll exactly WINDOW_DAYS old (1.0 today)
    "REFERENCE_SAMPLE": 800,  # a poll this size gets sample-size weight 1.0
    "SAMPLE_SIZE_EXPONENT": 0.5,  # weight = (n/ref)^0.5 (sampling error shrinks with sqrt n). 0 = ignore size
    "SAMPLE_SIZE_CAP": 2500,  # n above this gets no extra weight; stops huge online panels dominating
    "POPULATION_WEIGHTS": {"LV": 1.0, "RV": 0.85, "A": 0.7},  # Model 1's values. The team's blackboard ratio is
                                                             #   1 : 0.9 : 0.8 (used by Ozempic and DadBod)
    "RV_MARGIN_SHIFT": "estimate",  # MARKS: "estimate" = measured each run from 2026 surveys released for both LV and RV
                                    #   (average LV - RV margin, shrunk toward 0). A number = fixed shift added to RV polls
    "RV_SHIFT_PRIOR_PAIRS": 5,  # MARKS: the "no shift" assumption counts as this many pairs when estimating
    "RV_SHIFT_CLIP": (-3.0, 3.0),  # MARKS: estimated shift kept in this range
    "PARTISAN_WEIGHTS": {"Republican": 0.6, "Democrat": 0.6},  # party/campaign-sponsored polls count less
    "PARTISAN_HOUSE_EFFECTS": {"Republican": 3.0, "Democrat": 3.0},  # pts shifted AGAINST the sponsor
    "GENERIC_WEIGHT": 0.5,  # state poll asking "a Democrat or a Republican?" instead of naming candidates
    "BIGGER_FIELD_WEIGHT": 0.9,  # full-field version (3+ candidates offered); 1.0 = neutral
    # Pollster quality. CHANGED: grades come from the VoteHub scorecard instead of the spreadsheet's rank column
    "GRADE_TO_RANK": {"A+": 10, "A": 9, "B": 8, "C": 7, "D": 6, "F": 5},  # VoteHub grade -> Model 1's 5-10 scale
    "QUALITY_SCORE_MIN": 5,
    "QUALITY_SCORE_MAX": 10,
    "MIN_QUALITY_WEIGHT": 0.5,  # rank 5 counts half as much as rank 10
    "UNRATED_POLLSTERS": "random",  # NEW: pollsters VoteHub hasn't graded. "random" = in EVERY simulation each one gets
                                    #   the grade, past error and lean of a randomly drawn graded pollster (redrawn each
                                    #   simulation); "fixed" = Model 1's rule: UNRANKED_SCORE + DEFAULT_TYPICAL_ERROR, no lean
    "UNRANKED_SCORE": 5,  # "fixed" mode: rank given to ungraded pollsters (5 = lowest)
    # Typical error = pollster's mean absolute miss. CHANGED: filled from the scorecard's Percent_Error automatically
    "TYPICAL_ERROR_BY_POLLSTER": {},  # manual overrides, e.g. {"Emerson College": 3.4}
    "DEFAULT_TYPICAL_ERROR": 4.0,  # ungraded pollsters in "fixed" mode; also the reference for the accuracy weight
    "MAE_TO_SD": 1.25,  # for a normal distribution SD = 1.25 x mean absolute error
    "ACCURACY_WEIGHT_CLIP": (0.5, 2.0),  # NEW: accuracy weight kept in this range (real scorecard errors would otherwise
                                         #   let one very accurate firm count up to 10x)
    "FLOOD_EXPONENT": 0.5,  # each of a pollster's k polls in a state gets 1/k^0.5 (0 = no penalty, 1 = counts once)
    "SYSTEMATIC_POLL_ERROR_SD": 4.5,  # miss shared by all polls of a state; doesn't shrink with more polls

    # ---------------- House effects (pollster lean) ----------------
    "SCORECARD_HOUSE_EFFECTS": True,  # NEW: subtract each graded pollster's known lean (VoteHub scorecard). Skipped for
                                      #   partisan polls (they already get PARTISAN_HOUSE_EFFECTS)
    "HOUSE_EFFECT_MODE": "in_cycle",  # MARKS: "in_cycle" = measure each pollster's lean THIS year (its polls vs other
                                      #   pollsters' polls of the same states), shrunk toward its scorecard lean;
                                      #   "scorecard" = Pray's fixed VoteHub lean only. Tested on 1998-2022: in-cycle cut
                                      #   polling-average error ~6% and improved Brier (better in 10 of 15 cycles)
    "HOUSE_EFFECT_K": 5,  # MARKS: the scorecard lean (0 if ungraded) counts as this many polls when estimating the lean
    "HOUSE_EFFECT_ITERS": 3,  # MARKS: re-estimate this many times (each round re-corrects the comparison polls)
    "HOUSE_EFFECT_WINDOW_DAYS": 60,  # MARKS: estimate leans from polls this recent only (as in the 1998-2022 test); older
                                     #   polls compared with today's consensus would mistake the race's trend for a lean
    "ESTIMATE_HOUSE_EFFECTS": False,  # Model 1's data-driven estimate (residual vs state average, shrunk to 0). Off:
                                      #   with 1-2 polls per pollster it mostly chases noise
    "HOUSE_EFFECT_SHRINKAGE": 3.0,

    # ---------------- National environment ----------------
    "PRESIDENT_PARTY": "R",  # unpopular R president helps D
    "NAT_HALF_LIFE_DAYS": 30,  # approval moves slowly -> longer half-life than state polls
    "NAT_REFERENCE_SAMPLE": 1000,
    "NAT_POPULATION_WEIGHTS": {"LV": 1.0, "RV": 1.0, "Adults": 0.8},
    "NAT_PARTISAN_WEIGHT": 0.6,
    "NAT_NET_APPROVAL_SHIFT": {"LV": 0.0, "RV": 0.0, "Adults": 0.0},  # pts added to net approval by population
    "NAT_USE_QUALITY": True,  # same grade weighting for national polls (ungraded = UNRANKED_SCORE, not random)
    "ENV_SLOPE": 0.35,  # pts of Senate margin per pt of net approval
    "APPROVAL_NEUTRAL_NET": 0.0,  # net approval at which the environment is even
    "MIDTERM_PENALTY": 0.0,  # extra pts against the president's party beyond approval
    "USE_GENERIC_BALLOT": True,
    "GENERIC_BALLOT_BLEND": 0.75,  # MARKS (was 0.5): the generic ballot is already a margin, approval needs an uncalibrated
                                   #   slope -> lean on the generic ballot. env = blend x GB env + (1 - blend) x approval env
    "GB_LOOKBACK_DAYS": 90,  # VoteHub national coverage is thin after June, so look back further
    "GB_MIN_POLLS": 10,  # if fewer surveys than this in the window, use the N most recent
    "GB_POPULATION_PREFERENCE": ["LV", "RV", "A"],  # one survey released for several populations -> keep the most voter-like
    "GB_MARGIN_BASIS": "two_party",  # "two_party" = 100*(D-R)/(D+R); "raw" = D-R as reported
    "GB_APPLY_RV_SHIFT": True,
    "GB_HALF_LIFE_DAYS": 21,
    "GB_SHIFT": 0.0,  # pts added to the generic-ballot margin
    "NAT_KALSHI_POLL_EQUIV": 5,  # NEW: Kalshi's polling-average markets (RCP + VoteHub approval, VoteHub generic ballot)
                                 #   are blended with the VoteHub polls as if they were this many polls. 0 = ignore them
    "NAT_FRESH_HALF_LIFE_DAYS": 14,  # MARKS: when blending with Kalshi, VoteHub polls count as "fresh-poll equivalents"
                                     #   (a poll this many days old counts 1/2), so a stale feed can't outvote the market
    "NAT_MIX_LOOKBACK_DAYS": 90,  # NEW: window for the typical "neither" share (approval) and D+R share (generic ballot)
                                  #   used to put Kalshi's averages on the same scale as the polls (Step 5)
    "ENV_OVERRIDE": None,  # a number (e.g. 5.0 = D+5) bypasses all of the above

    # ---------------- Fundamentals prior ----------------
    "PVI_TO_MARGIN": 2.0,  # PVI is in vote share; 1 pt share = 2 pts margin
    "ENV_ELASTICITY_DEFAULT": 1.0,  # how much each state moves with the national environment
    "ENV_ELASTICITY": {},  # per-state overrides, e.g. {"New Hampshire": 1.2}
    "INCUMBENCY_ADVANTAGE": 2.0,  # pts for an elected incumbent on the ballot (calibration replaces this with the 2014-22 estimate)
    "APPOINTED_INCUMBENT_ADVANTAGE": 1.0,  # appointed incumbents get about half
    "STATE_PRIOR_ADJUSTMENTS": {},  # manual pts added to a state's prior, e.g. {"Texas": +2}
    "CANDIDATE_EFFECTS": True,  # MARKS: add each nominee's past over/under-performance (candidate_history.csv) to the prior
    "CANDIDATE_EFFECT_WEIGHT": 0.3,  # MARKS: share of a past over-performance that carries into this race (history: ~0.3)
    "CANDIDATE_EFFECT_CLIP": 25.0,  # MARKS: cap on any one candidate's past over-performance (pts)
    "CANDIDATE_MAX_AGE_YEARS": 8,  # MARKS: ignore races older than this
    "PRIOR_SD": 7.5,  # how far a race can stray from lean + environment
    "INDEPENDENT_PRIOR_SD_MULT": 1.3,  # PVI says less about races with an independent challenger -> wider prior

    # ---------------- Bayesian update ----------------
    "POLL_PRECISION_MULT": 1.0,  # >1 trusts polls more than the prior, <1 less
    "POLL_RELIABILITY": "bayesian",  # NEW: "bayesian" (DEFAULT) = Model 1: thin races are already trusted less through
                                     #   their wider poll uncertainty. A number k = ALSO multiply poll precision by
                                     #   n_eff / (n_eff + k) (DadBod's rule: half weight at k effective polls). Leans
                                     #   harder on fundamentals + Kalshi where polling is thin
    "KALSHI_SD": 8.0,  # MARKS (was 17): uncertainty (pts) of Kalshi's implied margin. 8 = roughly the prior's weight:
                       #   ~10-20% of a well-polled race, ~50% of an unpolled one. None or 0 = comparison only
    "KALSHI_CONVERT_SD": 4.7,  # MARKS: probability -> margin uses at least this SD (the historical final-weeks error of a
                               #   polled race); unpolled races use their own wider prior SD. "model" = Pray's old behaviour
    "KALSHI_LONGSHOT_SLOPE": 1.15,  # MARKS: favourite-longshot correction, P' = logistic(slope x logit P). 1 = none
    "KALSHI_USE_RANGE": (0.06, 0.94),  # MARKS: prices at the 1-5 cent floor only say "safe" (tick size, fees, longshot
                                       #   bias) -> not turned into a margin
    "CORRELATED_ERRORS": True,  # NEW: True = the prior, polls and Kalshi tend to miss in the same direction (Kalshi
                                #   traders read the same polls), so each race's range is wider. Weights are unchanged.
                                #   Independent of SCORECARD_HOUSE_EFFECTS: both, either or neither can be on
    "SOURCE_ERROR_CORR": 0.3,  # NEW: how strongly they miss together when CORRELATED_ERRORS is on (0-1)

    # ---------------- Time to election ----------------
    "DAILY_DRIFT_SD": 0.25,  # extra SD per sqrt(day) left; 0 on election day
    "DRIFT_SHAPE": "sqrt",  # "sqrt" (random walk) or "linear"

    # ---------------- Simulation error structure (MARKS) ----------------
    # "components": every simulated race = estimate + national miss + regional miss + lean-factor miss + state miss, each
    #   in points (sizes from history via calibration). Only the national miss has fat tails. "legacy" = Pray/Model 1's
    #   correlated Student-t (the CORR_* and T_DF settings below)
    "ERROR_MODEL": "components",
    "NATIONAL_ERROR_SD": 2.5,  # pts every race shares in a simulation (a national polling miss); history ~2.4
    "NATIONAL_T_DF": 8,  # fat tails for the national miss only (> 2; None = normal)
    "REGIONAL_ERROR_SD": 1.5,  # pts shared within a Census region; history ~1.6
    "LEAN_FACTOR_ERROR_SD": 1.0,  # pts of a miss that grows with partisanship (red and blue states miss in opposite directions)
    "LEAN_FACTOR_SCALE": 10,  # lean (vote-share pts) that gets the full lean-factor miss; capped at 2x
    "STATE_T_DF": 10,  # tails of the state-only miss (history ~10; None = normal)
    "MIN_STATE_ERROR_SD": 1.0,  # floor on the state-only miss (pts)

    # ---------------- Error distribution (ERROR_MODEL = "legacy" only) ----------------
    "USE_T_DIST": True,  # fat tails: upsets more common than a normal curve implies
    "T_DF": 5,  # degrees of freedom (> 2); lower = fatter tails
    "SHARED_TAIL_SCALE": True,  # one "bad polling year" draw shared by all states

    # ---------------- Between-state correlation (ERROR_MODEL = "legacy" only) ----------------
    "CORR_NATIONAL": 0.45,  # every pair shares this (national polling miss / wave)
    "CORR_REGION": 0.15,  # added if same Census region
    "CORR_LEAN": 0.15,  # added x exp(-|lean_i - lean_j| / LEAN_SCALE)
    "LEAN_SCALE": 10,
    "STATE_PAIR_CORR_OVERRIDES": {},  # exact correlation for specific pairs, e.g. {("Iowa", "Kansas"): 0.8}

    # ---------------- Race ratings ----------------
    "RATING_TOSSUP": 0.60, "RATING_LEAN": 0.75, "RATING_LIKELY": 0.95,  # favorite's win probability cut-offs
}

## SOURCES — web addresses and fallback tables

Plumbing, not assumptions. The `*_FALLBACK` tables are **only a check and a backup**: Kalshi markets and nominee names are found automatically each run (Step 1). These tables are used only for a race the search misses, or where Kalshi lists a party name instead of a candidate, and Model Testing flags any disagreement.

In [ ]:
SOURCES = {
    "VOTEHUB_URL": "https://api.votehub.com/polls",
    "KALSHI_URL": "https://api.elections.kalshi.com/trade-api/v2",
    "USER_AGENT": "DACSS-Team5-forecast",
    "KALSHI_SENATE_SERIES_PATTERN": r"(KX)?SENATE[A-Z]{2}S?",  # series searched for "<State> Senate winner?" events
    "KALSHI_CONTROL_SERIES": "CONTROLS",  # "Will Democrats win the U.S. Senate?"
    "KALSHI_CONTROL_FALLBACK": "CONTROLS-2026-D",
    # National polling-average markets: (signal, whose average, Kalshi series). The open event is found each run.
    "KALSHI_NATIONAL_SERIES": [("approval", "RCP", "KXTRUMPAPPROVE"), ("approval", "VoteHub", "KXTRUMPVH"),
                               ("generic_ballot", "VoteHub", "KXGENERICBALLOTVOTEHUB")],
    "KALSHI_UPDOWN_SD": 0.5,  # weekly "above X%?" markets: typical size (pts) of a week's move in a polling average
    "KALSHI_PROB_CLIP": (0.01, 0.99),  # keeps implied margins finite for 99%+ markets
    "KALSHI_HISTORY_START": "2026-06-01",  # first day of daily price history pulled for backtests
    "HTTP_TRIES": 4,  # MARKS: retries per request (Kalshi rate-limits bursts; Pray's search failed silently on these)
    "KALSHI_THROTTLE_S": 0.15,  # MARKS: pause between Kalshi requests

    # ---- FALLBACK / CHECK tables (checked Oct 26) ----
    "NOMINEES_FALLBACK": {  # (opposition, Republican); opposition None = highest-polling non-Republican in each poll
        "Alabama": ("Everett Wess", "Barry Moore"), "Alaska": ("Mary Peltola", "Dan Sullivan"),
        "Arkansas": ("Hallie Shoffner", "Tom Cotton"), "Colorado": ("John Hickenlooper", "Mark Baisley"),
        "Delaware": ("Chris Coons", "Michael Katz"), "Florida": ("Angie Nixon", "Ashley Moody"),
        "Georgia": ("Jon Ossoff", "Mike Collins"), "Idaho": (None, "Jim Risch"),
        "Illinois": ("Juliana Stratton", "Don Tracy"), "Iowa": ("Josh Turek", "Ashley Hinson"),
        "Kansas": ("Adam Hamilton", "Roger Marshall"), "Kentucky": ("Charles Booker", "Andy Barr"),
        "Louisiana": ("Jamie Davis", "Julia Letlow"), "Maine": ("Troy Jackson", "Susan Collins"),
        "Massachusetts": ("Ed Markey", "John Deaton"), "Michigan": ("Abdul El-Sayed", "Mike Rogers"),
        "Minnesota": ("Peggy Flanagan", "Michele Tafoya"), "Mississippi": ("Scott Colom", "Cindy Hyde-Smith"),
        "Montana": (None, "Kurt Alme"), "Nebraska": ("Dan Osborn", "Pete Ricketts"),
        "New Hampshire": ("Chris Pappas", "John Sununu"), "New Jersey": ("Cory Booker", "Justin Murphy"),
        "New Mexico": ("Ben Ray Lujan", "Larry Marker"), "North Carolina": ("Roy Cooper", "Michael Whatley"),
        "Ohio": ("Sherrod Brown", "Jon Husted"), "Oklahoma": ("N'Kiyla Thomas", "Kevin Hern"),
        "Oregon": ("Jeff Merkley", "David Brock Smith"), "Rhode Island": ("Jack Reed", "Raymond McKay"),
        "South Carolina": ("Annie Andrews", "Darline Graham"), "South Dakota": (None, "Mike Rounds"),
        "Tennessee": ("Marquita Bradshaw", "Bill Hagerty"), "Texas": ("James Talarico", "Ken Paxton"),
        "Virginia": ("Mark Warner", "Bert Mizusawa"), "West Virginia": ("Rachel Fetty Anderson", "Shelley Moore Capito"),
        "Wyoming": ("James Byrd", "Harriet Hageman"),
    },
    "KALSHI_EVENTS_FALLBACK": {  # Kalshi's tickers are inconsistent (Kentucky = SENATELA-26), hence the title search
        "Alabama": "SENATEAL-26", "Alaska": "SENATEAK-26", "Arkansas": "SENATEAR-26", "Colorado": "SENATECO-26",
        "Delaware": "SENATEDE-26", "Florida": "SENATEFLS-26", "Georgia": "SENATEGA-26", "Idaho": "SENATEID-26",
        "Illinois": "SENATEIL-26", "Iowa": "SENATEIA-26", "Kansas": "SENATEKS-26", "Kentucky": "SENATELA-26",
        "Louisiana": "KXSENATELA-26NOV", "Maine": "SENATEME-26", "Massachusetts": "SENATEMA-26",
        "Michigan": "SENATEMI-26", "Minnesota": "SENATEMN-26", "Mississippi": "SENATEMS-26", "Montana": "SENATEMT-26",
        "Nebraska": "SENATENE-26", "New Hampshire": "SENATENH-26", "New Jersey": "SENATENJ-26",
        "New Mexico": "SENATENM-26", "North Carolina": "SENATENC-26", "Ohio": "SENATEOHS-26",
        "Oklahoma": "SENATEOK-26", "Oregon": "SENATEOR-26", "Rhode Island": "SENATERI-26",
        "South Carolina": "SENATESC-26", "South Dakota": "SENATESD-26", "Tennessee": "SENATETN-26",
        "Texas": "SENATETX-26", "Virginia": "SENATEVA-26", "West Virginia": "SENATEWV-26", "Wyoming": "SENATEWY-26",
    },
}

### Shared settings from `run_all.py`
When `run_all.py` runs every model for a comparison, it makes them all use the same date (today's system date) and the same number of simulations (10,000) through two environment variables, which this cell reads. **When you run this notebook yourself, nothing is set and your CONFIG values are used.**

In [ ]:
if os.environ.get("FORECAST_AS_OF"):  # set only by run_all.py
    CONFIG["AS_OF_DATE"] = os.environ["FORECAST_AS_OF"]
if os.environ.get("FORECAST_N_SIMS"):
    CONFIG["N_SIMS"] = int(os.environ["FORECAST_N_SIMS"])
if os.environ.get("FORECAST_OUT_DIR"):  # MARKS: set only by final_forecast.py --test, so test runs never touch Output/
    CONFIG["OUT_DIR"] = Path(os.environ["FORECAST_OUT_DIR"])
if os.environ.get("FORECAST_AS_OF") or os.environ.get("FORECAST_N_SIMS"):
    print(f"run_all.py overrides: AS_OF_DATE = {CONFIG['AS_OF_DATE']}, N_SIMS = {CONFIG['N_SIMS']:,}")

### Calibration from past elections
`Final Model/calibration.ipynb` measures, from every competitive Senate race 1998–2022 (FiveThirtyEight's `raw_polls.csv`), how far polling averages and fundamentals actually missed, and writes the matching settings to `calibration.json` in this folder. With `USE_CALIBRATION = True` those values replace the CONFIG values above. The table shows every value it changed. Delete the file or set `USE_CALIBRATION = False` to go back to CONFIG.


In [ ]:
CALIBRATION = None
if CONFIG["USE_CALIBRATION"] and Path(CONFIG["CALIBRATION_FILE"]).exists():
    CALIBRATION = json.loads(Path(CONFIG["CALIBRATION_FILE"]).read_text())
    changed = []
    for k, v in CALIBRATION.get("overrides", {}).items():
        if k not in CONFIG:
            continue
        v = tuple(v) if isinstance(CONFIG[k], tuple) and isinstance(v, list) else v
        changed.append({"setting": k, "CONFIG value": CONFIG[k], "calibrated value": v})
        CONFIG[k] = v
    print(f"Calibration applied: {CALIBRATION.get('created', '?')} ({CALIBRATION.get('source', '')})")
    display(pd.DataFrame(changed).set_index("setting"))
else:
    print("No calibration applied: using CONFIG as written"
          + ("" if CONFIG["USE_CALIBRATION"] else " (USE_CALIBRATION = False)") + ".")


### Checks and the formula with your current settings
`validate_config` catches impossible settings before they produce silent nonsense. The formula below is rewritten with the numbers from CONFIG, so anyone can see exactly what this run does.

In [ ]:
def validate_config(cfg):
    """Catch impossible settings before they produce silent nonsense."""
    assert cfg["T_DF"] > 2, "T_DF must be > 2 (t variance is undefined otherwise)"
    assert cfg["CORR_NATIONAL"] + cfg["CORR_REGION"] + cfg["CORR_LEAN"] < 1, "correlations must leave room for state noise"
    assert 0 <= cfg["GENERIC_BALLOT_BLEND"] <= 1, "GENERIC_BALLOT_BLEND must be in [0, 1]"
    assert cfg["POLL_VERSION_MODE"] in {"split", "keep_all", "prefer_h2h", "prefer_full_field"}
    assert cfg["UNDECIDED_MODE"] in {"ignore", "proportional", "fixed_share", "against_incumbent"}
    assert cfg["DRIFT_SHAPE"] in {"sqrt", "linear"}
    assert cfg["RECENCY_MODE"] in {"half_life", "window"}, "RECENCY_MODE: 'half_life' or 'window'"
    assert cfg["UNRATED_POLLSTERS"] in {"random", "fixed"}, "UNRATED_POLLSTERS: 'random' or 'fixed'"
    assert cfg["POLL_RELIABILITY"] == "bayesian" or float(cfg["POLL_RELIABILITY"]) >= 0, "POLL_RELIABILITY: 'bayesian' or a number >= 0"
    assert cfg["KALSHI_SD"] in (None, 0) or cfg["KALSHI_SD"] > 0, "KALSHI_SD: a positive number, or None/0 to switch Kalshi off"
    assert cfg["KALSHI_CONVERT_SD"] == "model" or float(cfg["KALSHI_CONVERT_SD"]) > 0, "KALSHI_CONVERT_SD: a number or 'model'"
    assert cfg["KALSHI_LONGSHOT_SLOPE"] > 0
    assert 0 <= cfg["SOURCE_ERROR_CORR"] < 1 and cfg["NAT_KALSHI_POLL_EQUIV"] >= 0
    assert 0 <= cfg["WINDOW_OLDEST_WEIGHT"] <= 1 and cfg["WINDOW_DAYS"] > 0
    assert cfg["RATING_TOSSUP"] <= cfg["RATING_LEAN"] <= cfg["RATING_LIKELY"] <= 1
    assert cfg["HOLDOVER_D"] + cfg["HOLDOVER_R"] + len(cfg["SEATS_UP"]) == 100, "seat counts must total 100"
    assert cfg["RV_MARGIN_SHIFT"] == "estimate" or isinstance(cfg["RV_MARGIN_SHIFT"], (int, float)), "RV_MARGIN_SHIFT: 'estimate' or a number"
    assert cfg["ERROR_MODEL"] in {"components", "legacy"}, "ERROR_MODEL: 'components' or 'legacy'"
    for k in ("NATIONAL_T_DF", "STATE_T_DF"):
        assert cfg[k] is None or cfg[k] > 2, f"{k} must be > 2 or None"
    assert min(cfg["NATIONAL_ERROR_SD"], cfg["REGIONAL_ERROR_SD"], cfg["LEAN_FACTOR_ERROR_SD"], cfg["MIN_STATE_ERROR_SD"]) >= 0
    assert 0 <= cfg["CANDIDATE_EFFECT_WEIGHT"] <= 1, "CANDIDATE_EFFECT_WEIGHT must be in [0, 1]"
    assert cfg["HOUSE_EFFECT_MODE"] in {"in_cycle", "scorecard"}, "HOUSE_EFFECT_MODE: 'in_cycle' or 'scorecard'"
    assert cfg["HOUSE_EFFECT_K"] >= 0
    assert not (cfg["HOUSE_EFFECT_MODE"] == "in_cycle" and cfg["ESTIMATE_HOUSE_EFFECTS"]), "in-cycle house effects already replace ESTIMATE_HOUSE_EFFECTS"


def control_threshold(cfg):
    """Seats Democrats need for control (50 if a D VP breaks ties, 51 if R)."""
    if cfg["D_SEATS_FOR_CONTROL"] is not None:
        return cfg["D_SEATS_FOR_CONTROL"]
    return 50 if cfg["VP_PARTY"] == "D" else 51


def kalshi_on(cfg):
    return bool(cfg["KALSHI_SD"])


def render_formula(cfg):
    c = cfg
    ks = c["KALSHI_SD"]
    rel = c["POLL_RELIABILITY"]
    rec = (f"half-life {c['RECENCY_HALF_LIFE_DAYS']} days (all polls)" if c["RECENCY_MODE"] == "half_life"
           else f"last {c['WINDOW_DAYS']} days, weight 1.0 → {c['WINDOW_OLDEST_WEIGHT']}")
    conv = "the race's prior+polls SD" if c["KALSHI_CONVERT_SD"] == "model" else f"max({float(c['KALSHI_CONVERT_SD']):g}, the race's prior+polls SD)"
    err = (f"national {c['NATIONAL_ERROR_SD']:g} (t df {c['NATIONAL_T_DF']}) + regional {c['REGIONAL_ERROR_SD']:g} + lean factor "
           f"{c['LEAN_FACTOR_ERROR_SD']:g} + state (the rest; t df {c['STATE_T_DF']})" if c["ERROR_MODEL"] == "components"
           else f"legacy: Student-t df {c['T_DF']}; correlation national {c['CORR_NATIONAL']} + region {c['CORR_REGION']} + lean {c['CORR_LEAN']}")
    lines = [
        "**margin** = (π_F·prior + π_P·polls + π_K·Kalshi) / (π_F + π_P + π_K)",
        f"- π_F = 1 / {c['PRIOR_SD']:g}² (independent-led races: 1 / ({c['PRIOR_SD']:g}×{c['INDEPENDENT_PRIOR_SD_MULT']:g})²)",
        f"- π_P = {c['POLL_PRECISION_MULT']:g} / poll SD², poll SD = √(single-poll SD² / n_eff + {c['SYSTEMATIC_POLL_ERROR_SD']:g}² + undecided SD²)"
        + ("" if rel == "bayesian" else f", × n_eff / (n_eff + {rel}) (extra reliability discount)"),
        (f"- π_K = 1 / {ks:g}²; Kalshi margin = Φ⁻¹(P') × {conv}, P' = longshot-corrected price (slope {c['KALSHI_LONGSHOT_SLOPE']:g}); "
         f"used only for prices in {c['KALSHI_USE_RANGE'][0]:.0%}–{c['KALSHI_USE_RANGE'][1]:.0%}") if kalshi_on(c)
        else "- π_K = 0 (Kalshi shown for comparison only)",
        f"- prior = {c['PVI_TO_MARGIN']:g} × partisan lean + {c['ENV_ELASTICITY_DEFAULT']:g} × national environment ± incumbency "
        f"({c['INCUMBENCY_ADVANTAGE']:g} pts; appointed {c['APPOINTED_INCUMBENT_ADVANTAGE']:g})"
        + (f" + {c['CANDIDATE_EFFECT_WEIGHT']:g} × the nominees' past over-performance" if c["CANDIDATE_EFFECTS"] else ""),
        f"- national environment = {c['GENERIC_BALLOT_BLEND']:g} × generic ballot + {1 - c['GENERIC_BALLOT_BLEND']:g} × "
        f"{c['ENV_SLOPE']:g} × −(net approval); each = VoteHub polls (as fresh-poll equivalents) + Kalshi averages counted as "
        f"{c['NAT_KALSHI_POLL_EQUIV']:g} polls",
        f"- polls: recency {rec}; LV:RV:A = " + ":".join(f"{v:g}" for v in c["POPULATION_WEIGHTS"].values())
        + f"; RV shift {c['RV_MARGIN_SHIFT'] if isinstance(c['RV_MARGIN_SHIFT'], str) else format(c['RV_MARGIN_SHIFT'], '+g')}"
        + f"; LV version preferred {'on' if c['PREFER_LV_VERSION'] else 'off'}"
        + f"; flooding 1/k^{c['FLOOD_EXPONENT']}; partisan: weight {c['PARTISAN_WEIGHTS']['Republican']:g}, shift "
        f"{c['PARTISAN_HOUSE_EFFECTS']['Republican']:g} pts; house effects "
        + (f"in-cycle (shrunk to the scorecard lean, k = {c['HOUSE_EFFECT_K']:g})" if c["HOUSE_EFFECT_MODE"] == "in_cycle"
           else f"scorecard {'on' if c['SCORECARD_HOUSE_EFFECTS'] else 'off'}")
        + f"; ungraded pollsters {c['UNRATED_POLLSTERS']}",
        f"- uncertainty: source errors {'correlated (ρ = ' + str(c['SOURCE_ERROR_CORR']) + ')' if c['CORRELATED_ERRORS'] else 'independent'}"
        f"; drift {c['DAILY_DRIFT_SD']:g}/√day; simulation errors: {err}; {c['N_SIMS']:,} simulations"
        + (f"; backtest as of {c['AS_OF_DATE']}" if c["AS_OF_DATE"] else ""),
    ]
    display(Markdown("\n".join(lines)))


validate_config(CONFIG)
for k in ("OUT_DIR", "CACHE_DIR"):
    CONFIG[k].mkdir(parents=True, exist_ok=True)
print(f"{len(CONFIG['SEATS_UP'])} races | D needs {control_threshold(CONFIG)} seats "
      f"({control_threshold(CONFIG) - CONFIG['HOLDOVER_D']} wins) | repo root: {REPO_ROOT}")
render_formula(CONFIG)


## Step 1: Load the raw data

**In plain English:** download the latest polls and market prices. If a website is down, use the last copy saved in `Data_Cache/` and say so. Files are read once here; everything that depends on `CONFIG` happens inside `run_model()`, so Model Testing can re-run it with different settings without downloading again.

| Source | What | Saved copy |
|---|---|---|
| VoteHub `us-senator` | Every 2026 Senate general-election poll (state, pollster, dates, sample, results) | `votehub_senate_cache.csv` |
| VoteHub `generic-ballot` | National "which party for Congress?" polls | `votehub_generic_ballot_cache.csv` |
| VoteHub `approval` (Donald Trump) | Presidential approval polls | `votehub_approval_cache.csv` |
| Kalshi Senate markets | Every race's market, **found automatically** by title ("Maine Senate winner?"), with candidate names and prices, plus the Senate-control market | `kalshi_races_cache.csv` |
| Kalshi national markets | Today's polling-average markets: RCP approval, VoteHub approval, VoteHub generic ballot | `kalshi_national_cache.csv` |
| Kalshi price history | Daily closing prices, pulled only for backtests (`AS_OF_DATE`) | `kalshi_history_cache.csv` |
| `Data/` files | Partisan lean + region per state, VoteHub pollster scorecards | — |

**Why search Kalshi by title?** Kalshi's codes aren't consistent: Kentucky's race is filed under `SENATELA-26` (LA = Louisiana!) and Louisiana's under `KXSENATELA-26NOV`. Searching by title finds the right market regardless. The fallback table is used only for races the search misses.

**Matching poll candidates:** VoteHub lists candidates by name, not party, so each poll is matched to the nominees (from Kalshi's market names) by surname. The opposition is the highest-polling non-Republican nominee (the "credible independent" rule, e.g. Osborn in Nebraska). Polls listing "Democrat" / "Republican" instead of names are kept as **generic** polls. Polls missing a nominee (old hypothetical match-ups) are dropped.

In [ ]:
SOURCE_STATUS = {}  # source -> "live" or "cache (...)"; read by Model Testing and run_all.py


KALSHI_LOG = []  # MARKS: every Kalshi request that failed after all retries (shown in Model Testing)


def http_json(url, timeout=20):
    """GET a JSON API with retries. MARKS: Kalshi rate-limits bursts of requests (HTTP 429); Pray's search caught those
    errors silently, so a different set of races fell back to the hand table on every run. Now: a short pause before
    every Kalshi request, and up to HTTP_TRIES attempts with exponential back-off."""
    is_kalshi = "kalshi" in url
    for attempt in range(SOURCES["HTTP_TRIES"]):
        if is_kalshi:
            time.sleep(SOURCES["KALSHI_THROTTLE_S"])
        try:
            req = urllib.request.Request(url, headers={"User-Agent": SOURCES["USER_AGENT"], "Accept": "application/json"})
            with urllib.request.urlopen(req, timeout=timeout) as r:  # 20 s timeout so a dead API can't hang the notebook
                return json.load(r)
        except urllib.error.HTTPError as e:
            if e.code not in (429, 500, 502, 503, 504) or attempt == SOURCES["HTTP_TRIES"] - 1:
                raise
        except (urllib.error.URLError, TimeoutError, ConnectionError):
            if attempt == SOURCES["HTTP_TRIES"] - 1:
                raise
        time.sleep(0.5 * 2 ** attempt)  # 0.5, 1, 2 s


def kalshi_paged(path, key, params):
    """All pages of a Kalshi list endpoint (MARKS: follows the cursor instead of reading only the first page)."""
    out, cursor = [], None
    for _ in range(50):
        url = f"{SOURCES['KALSHI_URL']}{path}?{params}" + (f"&cursor={cursor}" if cursor else "")
        j = http_json(url)
        out += j.get(key) or []
        cursor = j.get("cursor")
        if not cursor:
            break
    return out


def with_cache(name, fetch, parse_dates):
    """Run fetch(); save to the cache on success, read the cache on failure."""
    path = CONFIG["CACHE_DIR"] / name
    try:
        df = fetch()
        if df is None or df.empty:
            raise ValueError("empty result")
        df.to_csv(path, index=False)
        SOURCE_STATUS[name] = "live"
        print(f"{name}: fetched {len(df)} rows live")
    except Exception as e:  # network down, API changed, ...
        if not path.exists():
            raise RuntimeError(f"{name}: API failed ({type(e).__name__}: {e}) and there is no saved copy") from e
        df = pd.read_csv(path)
        SOURCE_STATUS[name] = f"cache ({type(e).__name__})"
        print(f"{name}: API unavailable ({type(e).__name__}); using saved copy with {len(df)} rows")
    for c in parse_dates:
        if c in df.columns:
            df[c] = pd.to_datetime(df[c], utc=True).dt.tz_localize(None).dt.normalize()
    return df


# ---------------- VoteHub ----------------
def fetch_senate():
    rows = []
    for p in http_json(f"{SOURCES['VOTEHUB_URL']}?poll_type=us-senator"):
        subj = str(p.get("subject") or "")
        if not subj.startswith("2026 ") or subj.endswith((" Democratic", " Republican")):
            continue  # 2026 general-election polls only (drop primaries)
        state = subj[5:]
        if state not in CONFIG["SEATS_UP"]:
            continue
        rows.append({"id": p.get("id"), "end_date": p.get("end_date"), "created_at": p.get("created_at"),
                     "state": state, "pollster": p.get("pollster"), "sample_size": p.get("sample_size"),
                     "population": (p.get("population") or "").lower(), "partisan": p.get("partisan") or "",
                     "internal": bool(p.get("internal")),
                     "answers": "|".join(f"{a['choice']}={a['pct']}" for a in p.get("answers", []))})
    return pd.DataFrame(rows)


NAT_COLS = {"generic-ballot": ["Dem", "Rep"], "approval": ["Approve", "Disapprove"]}


def fetch_national(poll_type, keys, extra=""):
    rows = []
    for p in http_json(f"{SOURCES['VOTEHUB_URL']}?poll_type={poll_type}{extra}"):
        ans = {a["choice"].strip().lower(): a["pct"] for a in p.get("answers", [])}
        vals = [next((v for k, v in ans.items() if k.startswith(key)), None) for key in keys]
        if None in vals:
            continue
        rows.append({"Date": p.get("end_date"), "Released": p.get("created_at"), "Pollster": p.get("pollster"),
                     **dict(zip(NAT_COLS[poll_type], vals)), "SampleSize": p.get("sample_size"),
                     "Population": (p.get("population") or "").upper(),
                     "Partisan": bool(p.get("partisan") or p.get("internal"))})
    return pd.DataFrame(rows)


# ---------------- Kalshi ----------------
def kalshi_mid(m):
    """Mid price (0-1) of a Kalshi market; handles *_dollars (new) and cents (old) fields."""
    def f(k):
        v = m.get(k + "_dollars")
        if v not in (None, ""):
            return float(v)
        v = m.get(k)
        return None if v in (None, "") else float(v) / 100
    bid, ask, last = f("yes_bid"), f("yes_ask"), f("last_price")
    if bid and ask and ask >= bid:
        return (bid + ask) / 2
    return last


def is_live_market(m):
    return str(m.get("status", "active")).lower() in ("active", "open", "initialized", "unopened")


def fetch_kalshi_races():
    """Find each 2026 Senate race on Kalshi by its TITLE, not its ticker. Fallback table only for races the search misses."""
    today = pd.Timestamp.today().strftime("%Y-%m-%d")
    K = SOURCES["KALSHI_URL"]
    series = [s["ticker"] for s in kalshi_paged("/series", "series", "category=Elections&tags=Senate")
              if re.fullmatch(SOURCES["KALSHI_SENATE_SERIES_PATTERN"], s["ticker"])]
    found = {}
    for ser in series:
        try:
            events = kalshi_paged("/events", "events", f"series_ticker={ser}&status=open&with_nested_markets=true")
        except Exception as e:
            KALSHI_LOG.append({"request": f"events {ser}", "error": f"{type(e).__name__}: {e}"})
            continue
        for ev in events:
            m = re.fullmatch(r"(.+?) Senate winner\?", str(ev.get("title") or "").strip())
            is_2026 = "2026" in str(ev.get("sub_title") or "") or "-26" in ev.get("event_ticker", "")
            if not m or not is_2026 or m.group(1).strip() not in CONFIG["SEATS_UP"]:
                continue
            mk = [x for x in ev.get("markets", []) if is_live_market(x)]
            if not any(x["ticker"].endswith("-R") for x in mk):
                continue
            vol = sum(float(x.get("volume_fp") or x.get("volume") or 0) for x in mk)
            st = m.group(1).strip()
            if st not in found or vol > found[st][0]:  # two events for one race -> keep the busier one
                found[st] = (vol, ev, mk, "search")
    for st, ev_t in SOURCES["KALSHI_EVENTS_FALLBACK"].items():  # races the search missed
        if st not in found:
            try:
                j = http_json(f"{K}/events/{ev_t}?with_nested_markets=true")
                ev = j.get("event", {})
                mk = [x for x in (j.get("markets") or ev.get("markets", [])) if is_live_market(x)]
                found[st] = (0, {**ev, "event_ticker": ev_t}, mk, "fallback table")
            except Exception as e:
                KALSHI_LOG.append({"request": f"fallback {st} {ev_t}", "error": f"{type(e).__name__}: {e}"})
    rows = []
    for st, (_, ev, mk, how) in found.items():
        for x in mk:
            rows.append({"State": st, "event_ticker": ev.get("event_ticker"), "event_title": ev.get("title"),
                         "market_ticker": x["ticker"], "side": "R" if x["ticker"].endswith("-R") else "OPP",
                         "name": x.get("yes_sub_title") or x.get("subtitle"), "p_mid": kalshi_mid(x),
                         "found_by": how, "fetched": today})
    ctrl = None  # Senate-control market
    try:
        for ev in http_json(f"{K}/events?series_ticker={SOURCES['KALSHI_CONTROL_SERIES']}&status=open"
                            f"&with_nested_markets=true").get("events", []):
            if "2026" in ev.get("event_ticker", "") or "2026" in str(ev.get("sub_title") or ""):
                ctrl = next((x for x in ev.get("markets", []) if "democ" in str(x.get("yes_sub_title", "")).lower()), ctrl)
    except Exception as e:
        KALSHI_LOG.append({"request": "control market search", "error": f"{type(e).__name__}: {e}"})
    if ctrl is None:
        ctrl = http_json(f"{K}/markets/{SOURCES['KALSHI_CONTROL_FALLBACK']}")["market"]
    rows.append({"State": "CONTROL", "event_ticker": ctrl.get("event_ticker"), "event_title": ctrl.get("title"),
                 "market_ticker": ctrl["ticker"], "side": "D", "name": ctrl.get("yes_sub_title"), "p_mid": kalshi_mid(ctrl),
                 "found_by": "search", "fetched": today})
    return pd.DataFrame(rows)


def fetch_kalshi_national():
    """Today's polling-average markets. In each series, the open event closing soonest = the current reading."""
    today = pd.Timestamp.today().strftime("%Y-%m-%d")
    rows = []
    for signal, source, ser in SOURCES["KALSHI_NATIONAL_SERIES"]:
        try:
            events = kalshi_paged("/events", "events", f"series_ticker={ser}&status=open&with_nested_markets=true")
        except Exception as e:
            KALSHI_LOG.append({"request": f"national {ser}", "error": f"{type(e).__name__}: {e}"})
            continue
        events = [e for e in events if e.get("markets")]
        if not events:
            continue
        ev = min(events, key=lambda e: min(str(m.get("close_time") or "9") for m in e["markets"]))
        for m in ev["markets"]:
            rows.append({"signal": signal, "source": source, "series_ticker": ser, "event_ticker": ev.get("event_ticker"),
                         "market_ticker": m["ticker"], "strike_type": m.get("strike_type"), "floor_strike": m.get("floor_strike"),
                         "cap_strike": m.get("cap_strike"), "p_mid": kalshi_mid(m), "fetched": today})
    return pd.DataFrame(rows)


def candle_value(c, side):
    d = c.get(side) or {}
    if d.get("close_dollars") not in (None, ""):
        return float(d["close_dollars"])
    return None if d.get("close") in (None, "") else float(d["close"]) / 100


def fetch_kalshi_history():
    """Daily closes of every race's Republican market + the control market (one request per market)."""
    start = int(pd.Timestamp(SOURCES["KALSHI_HISTORY_START"], tz="UTC").timestamp())
    end = int(pd.Timestamp.now(tz="UTC").timestamp())
    k = DATA["kalshi_races"]
    rows = []
    for m in k.loc[k["side"].isin(["R", "D"]), "market_ticker"]:
        j = http_json(f"{SOURCES['KALSHI_URL']}/series/{m.split('-')[0]}/markets/{m}/candlesticks"
                      f"?start_ts={start}&end_ts={end}&period_interval=1440")
        for c in j.get("candlesticks", []):
            rows.append({"market_ticker": m, "end_period_ts": c["end_period_ts"], "yes_bid": candle_value(c, "yes_bid"),
                         "yes_ask": candle_value(c, "yes_ask"), "last": candle_value(c, "price")})
    return pd.DataFrame(rows)


def load_static(cfg):
    d = cfg["DATA_DIR"]
    leans = pd.read_csv(d / cfg["LEANS_FILE"], encoding="utf-8-sig")  # utf-8-sig strips the byte-order mark
    leans = leans[leans["State"] != "Source"].copy()  # drop the citation row
    leans["State"] = leans["State"].str.strip()
    leans["PartisanLean"] = pd.to_numeric(leans["PartisanLean"])
    cards = pd.read_csv(d / cfg["SCORECARD_FILE"])
    he = cards["House_Effect"].astype(str).str.strip()
    cards["house_effect_d"] = he.str[2:].astype(float) * np.where(he.str.startswith("D"), 1.0, -1.0)  # + = leans D
    return {"leans": leans.set_index("State"), "scorecards": cards}


DATA = {
    **load_static(CONFIG),
    "senate": with_cache("votehub_senate_cache.csv", fetch_senate, ["end_date", "created_at"]),
    "generic": with_cache("votehub_generic_ballot_cache.csv", lambda: fetch_national("generic-ballot", ["dem", "rep"]),
                          ["Date", "Released"]),
    "approval": with_cache("votehub_approval_cache.csv",
                           lambda: fetch_national("approval", ["approve", "disapprove"], "&subject=Donald%20Trump"),
                           ["Date", "Released"]),
    "kalshi_races": with_cache("kalshi_races_cache.csv", fetch_kalshi_races, []),
    "kalshi_national": with_cache("kalshi_national_cache.csv", fetch_kalshi_national, []),
}
if "found_by" not in DATA["kalshi_races"].columns:  # saved copies written before this column existed
    DATA["kalshi_races"]["found_by"] = "saved copy"

### Nominees, Kalshi check, and the poll table
Nominees come from Kalshi's candidate names (the fallback table only where Kalshi lists a party). Each VoteHub poll is then turned into one row with Model 1's columns: `Dem%` (the opposition nominee, incl. a credible independent), `Rep%`, `LVorRV`, `Partisan`, `Generic`, `Bigger_Field` (3+ candidates offered), plus the pollster's VoteHub grade, past error and lean.

In [ ]:
def surname(name):
    s = unicodedata.normalize("NFKD", str(name)).encode("ascii", "ignore").decode().lower()
    s = re.sub(r"[^a-z\- ]", "", s).strip()
    return s.split()[-1] if s else ""


def build_nominees(data):
    """{state: (republican, [opposition])} from Kalshi's market names; the fallback table only where Kalshi lists a party."""
    k = data["kalshi_races"]
    generic = lambda n: (not isinstance(n, str)) or "party" in n.lower()
    out, check = {}, []
    for st in CONFIG["SEATS_UP"]:
        g = k[k["State"] == st]
        hand_opp, hand_rep = SOURCES["NOMINEES_FALLBACK"][st]
        rep_k = [n for n in g.loc[g["side"] == "R", "name"] if not generic(n)]
        opp_k = [n for n in g.loc[g["side"] == "OPP", "name"] if not generic(n)]
        rep = rep_k[0] if rep_k else hand_rep
        opp = opp_k if opp_k else ([hand_opp] if hand_opp else [])
        source = "Kalshi" if rep_k and opp_k else ("fallback table" if g.empty else "Kalshi + fallback table (Kalshi lists parties)")
        agree = surname(rep) == surname(hand_rep) and (hand_opp is None or surname(hand_opp) in {surname(n) for n in opp})
        out[st] = (rep, opp)
        check.append({"State": st, "republican": rep, "opposition": ", ".join(opp) or "(top non-R in each poll)",
                      "source": source, "matches_fallback_table": agree})
    return out, pd.DataFrame(check).set_index("State")


NOMINEES, NOMINEE_CHECK = build_nominees(DATA)
_kr = DATA["kalshi_races"]
KALSHI_CHECK = pd.DataFrame({
    "found_event": _kr[_kr["side"] == "R"].set_index("State")["event_ticker"].reindex(sorted(CONFIG["SEATS_UP"])),
    "found_by": _kr[_kr["side"] == "R"].set_index("State")["found_by"].reindex(sorted(CONFIG["SEATS_UP"])),
    "fallback_table": pd.Series(SOURCES["KALSHI_EVENTS_FALLBACK"]),
})
KALSHI_CHECK["matches_fallback_table"] = KALSHI_CHECK["found_event"] == KALSHI_CHECK["fallback_table"]


def parse_answers(s):
    out = []
    for part in str(s).split("|"):
        if "=" in part:
            name, pct = part.rsplit("=", 1)
            out.append((name.strip(), float(pct)))
    return out


def party_answer(name):
    """'D' / 'R' if a poll answer is a party (generic ballot), else None."""
    n = name.lower()
    if re.search(r"\bdemocrat", n) and not re.search(r"\brepublican", n):
        return "D"
    if re.search(r"\brepublican", n) and not re.search(r"\bdemocrat", n):
        return "R"
    return None


def match_poll(row):
    """(dem_pct, rep_pct, n_candidates, generic) or None. 'dem' = the opposition (incl. a credible independent)."""
    ans = parse_answers(row["answers"])
    parties = {party_answer(n): p for n, p in ans if party_answer(n)}
    if "D" in parties and "R" in parties and len(parties) == len(ans):  # every answer is a party -> generic poll
        return parties["D"], parties["R"], len(ans), True
    rep_name, opp_names = NOMINEES[row["state"]]
    rep = next((p for n, p in ans if surname(n) == surname(rep_name)), None)
    if rep is None:
        return None
    others = [(n, p) for n, p in ans if surname(n) != surname(rep_name)]
    if opp_names:
        keys = {surname(n) for n in opp_names}
        opp = max((p for n, p in others if surname(n) in keys), default=None)  # highest-polling opposition nominee
    else:
        opp = max((p for _, p in others), default=None)
    return None if opp is None else (opp, rep, len(ans), False)


def scorecard_lookup(cards):
    """pollster name -> (VoteHub grade, % error, house effect); unknown pollsters -> (None, nan, nan)."""
    key = lambda s: re.sub(r"[^a-z0-9]", "", str(s).lower())
    lut = {}
    for _, r in cards.iterrows():
        names = [r["Scorecard_Name"]] + (str(r["Aliases"]).split("|") if pd.notna(r["Aliases"]) else [])
        for n in names:
            lut[key(n)] = (r["Grade"], float(r["Percent_Error"]), float(r["house_effect_d"]))
    return lambda p: lut.get(key(p), (None, np.nan, np.nan))


def build_poll_table(data):
    """VoteHub state polls -> one row per poll version, in Model 1's column names."""
    raw = data["senate"]
    m = raw.apply(match_poll, axis=1) if len(raw) else pd.Series(dtype=object)
    keep = m.notna().to_numpy(dtype=bool)
    p, m = raw[keep].copy(), m[keep]
    party = p["partisan"].fillna("").astype(str).str.upper()
    look = p["pollster"].map(scorecard_lookup(data["scorecards"]))
    out = pd.DataFrame({
        "Date": p["end_date"], "Released": p["created_at"] if "created_at" in p else pd.NaT,
        "State": p["state"], "Pollster": p["pollster"].astype(str),
        "Dem%": [x[0] for x in m], "Rep%": [x[1] for x in m], "n_cands": [x[2] for x in m], "Generic": [x[3] for x in m],
        "SampleSize": pd.to_numeric(p["sample_size"], errors="coerce"),
        "LVorRV": p["population"].fillna("").astype(str).str.upper().replace({"ADULTS": "A", "V": "RV"}),
        "Partisan": np.where(party.str.startswith("REP"), "Republican", np.where(party.str.startswith("DEM"), "Democrat", None)),
        "Grade": [x[0] for x in look], "Scorecard Error": [x[1] for x in look], "Scorecard Lean": [x[2] for x in look],
    })
    out["Partisan"] = out["Partisan"].astype("string")
    out["Bigger_Field"] = out["n_cands"] > 2
    out["Unrated"] = out["Grade"].isna()
    return out.reset_index(drop=True), int((~keep).sum())


DATA["polls"], N_UNMATCHED = build_poll_table(DATA)
print(f"Kalshi: {KALSHI_CHECK['found_event'].notna().sum()}/35 races found "
      f"({(KALSHI_CHECK['found_by'] == 'search').sum()} by title search); "
      f"nominees from Kalshi for {(NOMINEE_CHECK['source'] == 'Kalshi').sum()}/35 races")
print(f"VoteHub state polls: {len(DATA['senate'])} general-election rows -> {len(DATA['polls'])} with the nominees "
      f"({N_UNMATCHED} dropped: old hypothetical match-ups); {DATA['polls']['Unrated'].sum()} from ungraded pollsters")

## Step 2: Clean the polls

- **Pollster names:** fixes from `POLLSTER_NAME_FIXES`, then spellings that differ only by case/punctuation are merged (otherwise the flooding penalty would treat "Morning consult" and "Morning Consult" as two firms).
- **Duplicates** (`DROP_EXACT_DUPLICATES`): identical rows listed twice are dropped.
- **As-of filter** (backtests): only polls **released** by `AS_OF_DATE` (VoteHub's `created_at`), so nothing published later leaks in.
- **Age filter:** `MAX_POLL_AGE_DAYS`, or the `WINDOW_DAYS` window when `RECENCY_MODE = "window"`.
- **LV first** (`PREFER_LV_VERSION`, MARKS): a survey released for both likely and registered voters keeps only the likely-voter version (the population closest to who votes), instead of counting each version half.
- **Versions** (`POLL_VERSION_MODE`): one survey released several ways (e.g. likely and registered voters, or head-to-head and full field) counts once in total by default. A generic version is dropped when a named-candidate version of the same survey exists.

In [ ]:
def normalize_pollsters(names, fixes):
    """Apply manual fixes, then merge spellings that differ only by case/punctuation."""
    s = names.astype(str).str.strip().replace(fixes)
    key = s.str.lower().str.replace(r"[^a-z0-9]", "", regex=True)
    canonical = s.groupby(key).agg(lambda x: x.value_counts().index[0])  # most common spelling per key
    return key.map(canonical)


def resolve_today(cfg):
    """Model 'as of' date: AS_OF_DATE if set, otherwise today."""
    return pd.Timestamp(cfg["AS_OF_DATE"]) if cfg["AS_OF_DATE"] else pd.Timestamp.today().normalize()


def is_backtest(today):
    return today < pd.Timestamp.today().normalize()


def clean_polls(raw, cfg, today, verbose=False):
    p = raw.copy()
    if p.empty:
        return p
    p["Pollster"] = normalize_pollsters(p["Pollster"], cfg["POLLSTER_NAME_FIXES"])
    p["SampleSize"] = p["SampleSize"].fillna(cfg["MISSING_SAMPLE_SIZE"]).replace(0, cfg["MISSING_SAMPLE_SIZE"])
    n0 = len(p)
    if cfg["DROP_EXACT_DUPLICATES"]:
        p = p.drop_duplicates(subset=["Date", "State", "Pollster", "Dem%", "Rep%", "SampleSize", "LVorRV", "Bigger_Field", "Generic"])
    p = p[p["Date"] <= today]  # as-of filter on the field date
    p = p[p["Released"].isna() | (p["Released"] <= today)]  # ... and on the release date (backtests)
    p = p[p["SampleSize"] >= cfg["MIN_SAMPLE_SIZE"]]
    age = (today - p["Date"]).dt.days
    if cfg["MAX_POLL_AGE_DAYS"] is not None:
        p = p[age.loc[p.index] <= cfg["MAX_POLL_AGE_DAYS"]]
    if cfg["RECENCY_MODE"] == "window":
        p = p[(today - p["Date"]).dt.days <= cfg["WINDOW_DAYS"]]
    if verbose:
        print(f"Polls: {n0} matched -> {len(p)} after cleaning/filters"
              + (f" (window: last {cfg['WINDOW_DAYS']} days)" if cfg["RECENCY_MODE"] == "window" else ""))
    return p.reset_index(drop=True)


def apply_version_mode(p, cfg, verbose=False):
    """Decide how multi-version surveys (same pollster/state/end date) count."""
    p = p.copy()
    if p.empty:
        p["family"], p["n_versions"], p["w_version"] = [], [], []
        return p
    p["family"] = p["Date"].dt.strftime("%Y-%m-%d") + " | " + p["State"] + " | " + p["Pollster"]  # survey id
    n_before = len(p)
    if cfg["DROP_GENERIC_IF_CANDIDATE_VERSION"]:
        has_named = p.groupby("family")["Generic"].transform(lambda g: (~g).any())
        p = p[~(p["Generic"] & has_named)]
    if cfg["PREFER_LV_VERSION"]:  # MARKS: LV + RV releases of one survey -> keep the LV version only
        has_lv = p.groupby("family")["LVorRV"].transform(lambda g: (g == "LV").any())
        p = p[(p["LVorRV"] == "LV") | ~has_lv]
    mode = cfg["POLL_VERSION_MODE"]
    if mode in ("prefer_h2h", "prefer_full_field"):
        want = mode == "prefer_full_field"
        has_want = p.groupby("family")["Bigger_Field"].transform(lambda g: (g == want).any())
        p = p[(p["Bigger_Field"] == want) | ~has_want]
    p["n_versions"] = p.groupby("family")["family"].transform("size")
    p["w_version"] = 1.0 if mode == "keep_all" else 1.0 / p["n_versions"]  # each survey sums to weight 1
    if verbose:
        multi = (p.groupby("family").size() > 1).sum()
        print(f"Versions ({mode}): {n_before} rows -> {len(p)} rows in {p['family'].nunique()} surveys "
              f"({multi} surveys still have >1 version)")
    return p.reset_index(drop=True)

## Step 3: Undecided voters → poll margins

`other = 100 − Dem% − Rep%`. Only part of it is undecided (`OTHER_TO_UNDECIDED_*`); the rest is third-party support that stays put. Undecideds are allocated per `UNDECIDED_MODE`, and their uncertain split widens each poll's error by `2 × undecided × UNDECIDED_SPLIT_SD`. *(Unchanged from Model 1.)*

In [ ]:
def compute_margins(p, cfg):
    p = p.copy()
    if p.empty:
        for c in ("undecided", "margin", "undecided_sd"):
            p[c] = []
        return p
    d, r = p["Dem%"].astype(float), p["Rep%"].astype(float)
    other = (100 - d - r).clip(lower=0)
    frac = np.where(p["Bigger_Field"], cfg["OTHER_TO_UNDECIDED_FULL_FIELD"], cfg["OTHER_TO_UNDECIDED_H2H"])
    u = other * frac  # truly undecided share
    raw = d - r
    mode = cfg["UNDECIDED_MODE"]
    if mode == "ignore":
        m = raw
    elif mode == "proportional":
        m = raw * (d + r + u) / (d + r)  # undecideds break like decided voters
    elif mode == "fixed_share":
        m = raw + u * (2 * cfg["UNDECIDED_D_SHARE"] - 1)
    else:  # "against_incumbent"
        seat = p["State"].map(cfg["SEATS_UP"])
        status = p["State"].map(cfg["INCUMBENT_STATUS"]).fillna(cfg["DEFAULT_INCUMBENT_STATUS"])
        c = cfg["UNDECIDED_CHALLENGER_SHARE"]
        d_share = np.where(status == "open", 0.5, np.where(seat == "D", 1 - c, c))
        m = raw + u * (2 * d_share - 1)
    p["undecided"] = u
    p["margin"] = m
    p["undecided_sd"] = 2 * u * cfg["UNDECIDED_SPLIT_SD"]
    return p

## Step 4: Weight the polls and build state averages

Each poll's weight is the **product** of: recency, sample size, population (LV/RV/A), partisan sponsor, generic ballot, full field, pollster quality, historical accuracy, flooding, and survey-version share (Model 1's list). What changed:

| Factor | Model 1 | Pray_For_The_Floor |
|---|---|---|
| Recency | half-life 21 days | same by default; `RECENCY_MODE = "window"` switches to the 28-day linear window |
| Quality | spreadsheet rank column (5–10) | VoteHub grade → 5–10 (`GRADE_TO_RANK`) |
| Accuracy | blank column → everyone 4.0 | VoteHub's past error per pollster, weight clipped to `ACCURACY_WEIGHT_CLIP` |
| Ungraded pollsters | rank 5 + 4.0 error, fixed | `"random"`: a randomly drawn graded pollster's grade, error and lean, **redrawn in every simulation** (Step 7) |

**RV shift (MARKS).** Pray moved registered-voter polls 0.5 pt toward Republicans by assumption. Here the shift is **measured every run** (`RV_MARGIN_SHIFT = "estimate"`): every 2026 survey VoteHub lists for both likely and registered voters (Senate and generic ballot) gives one LV − RV gap; the average gap, shrunk toward 0 as if `RV_SHIFT_PRIOR_PAIRS` extra pairs showed no gap, is added to RV-only polls.

**In-cycle house effects (MARKS).** Pray subtracted each graded pollster's VoteHub scorecard lean, which describes how it missed in *past* elections. Marks measures each pollster's lean *this year*: every non-partisan poll from the last `HOUSE_EFFECT_WINDOW_DAYS` (60) days is compared with the same state's weighted average **without that pollster's polls**, and the pollster's lean is

$$\text{lean} = \frac{\text{sum of its residuals} + k \times \text{scorecard lean}}{n + k}, \qquad k = \texttt{HOUSE\_EFFECT\_K}\;(5)$$

so a firm with 2 polls stays near its scorecard lean and one with 16 is measured mostly from this year. Ungraded firms start from 0 in the tables and from their random draw in each simulation. The estimate is repeated 3 times so the comparison polls are corrected too. On 1998–2022 Senate races this cut the error of the final polling averages from 4.72 to 4.43 pts and improved Brier scores (better in 10 of 15 cycles). It **replaces** the scorecard subtraction rather than adding to it. `HOUSE_EFFECT_MODE = "scorecard"` restores Pray.

**Margin adjustments** (they move a poll's margin, not its weight): RV shift, partisan sponsor shift, and (NEW) each graded pollster's **scorecard lean** (`SCORECARD_HOUSE_EFFECTS`, skipped for partisan polls). Model 1's own data-driven estimate (`ESTIMATE_HOUSE_EFFECTS`) is still available and off by default. Turning both on would correct twice.

**Poll uncertainty per state** (Model 1): √(typical single-poll SD² ÷ n_eff + `SYSTEMATIC_POLL_ERROR_SD`² + undecided SD²), where n_eff is the effective number of polls. A single poll's SD is the larger of its sampling error and 1.25 × its pollster's past error.

**Ungraded pollsters in the tables:** with `"random"`, tables show the *average* over the graded pool (average quality, accuracy and lean). The simulations use a fresh draw each time.

In [ ]:
def grade_pool(data, cfg):
    """Quality weight, typical error and lean of every VoteHub-graded pollster: the pool random grades are drawn from."""
    c = data["scorecards"]
    rank = c["Grade"].map(cfg["GRADE_TO_RANK"]).fillna(cfg["UNRANKED_SCORE"]).to_numpy(dtype=float)
    lo, hi = cfg["QUALITY_SCORE_MIN"], cfg["QUALITY_SCORE_MAX"]
    q = cfg["MIN_QUALITY_WEIGHT"] + (1 - cfg["MIN_QUALITY_WEIGHT"]) * (np.clip(rank, lo, hi) - lo) / (hi - lo)
    return {"quality": q, "mae": c["Percent_Error"].to_numpy(dtype=float), "lean": c["house_effect_d"].to_numpy(dtype=float)}


def accuracy_parts(samp_sd, mae, cfg):
    """(poll error SD, accuracy weight) for sampling SD + pollster typical error; works on arrays of any shape."""
    pes = np.maximum(samp_sd, cfg["MAE_TO_SD"] * mae)
    w = np.clip((cfg["MAE_TO_SD"] * cfg["DEFAULT_TYPICAL_ERROR"] / pes) ** 2, *cfg["ACCURACY_WEIGHT_CLIP"])
    return pes, w


def weight_polls(p, cfg, today, data):
    p = p.copy()
    if p.empty:
        return p
    age = (today - p["Date"]).dt.days.clip(lower=0)
    p["age"] = age
    if cfg["RECENCY_MODE"] == "half_life":
        p["w_recency"] = 0.5 ** (age / cfg["RECENCY_HALF_LIFE_DAYS"])  # halves every half-life
    else:
        p["w_recency"] = 1 - (1 - cfg["WINDOW_OLDEST_WEIGHT"]) * age.clip(upper=cfg["WINDOW_DAYS"]) / cfg["WINDOW_DAYS"]
    n = p["SampleSize"].clip(upper=cfg["SAMPLE_SIZE_CAP"])
    p["w_size"] = (n / cfg["REFERENCE_SAMPLE"]) ** cfg["SAMPLE_SIZE_EXPONENT"]
    p["w_type"] = p["LVorRV"].map(cfg["POPULATION_WEIGHTS"]).fillna(1.0)
    p["w_partisan"] = p["Partisan"].map(cfg["PARTISAN_WEIGHTS"]).astype(float).fillna(1.0)
    p["w_generic"] = np.where(p["Generic"], cfg["GENERIC_WEIGHT"], 1.0)
    p["w_field"] = np.where(p["Bigger_Field"], cfg["BIGGER_FIELD_WEIGHT"], 1.0)
    k = p.groupby(["State", "Pollster"])["family"].transform("nunique")  # distinct surveys by this firm in this state
    p["w_flood"] = 1 / k ** cfg["FLOOD_EXPONENT"]
    # w_version set in apply_version_mode

    # quality + accuracy: graded pollsters from the scorecard; ungraded per UNRATED_POLLSTERS
    lo, hi = cfg["QUALITY_SCORE_MIN"], cfg["QUALITY_SCORE_MAX"]
    unr = p["Unrated"].to_numpy(dtype=bool)
    rank = p["Grade"].map(cfg["GRADE_TO_RANK"]).fillna(cfg["UNRANKED_SCORE"]).clip(lo, hi)
    p["w_quality"] = cfg["MIN_QUALITY_WEIGHT"] + (1 - cfg["MIN_QUALITY_WEIGHT"]) * (rank - lo) / (hi - lo)
    pd_, pr_ = p["Dem%"] / 100, p["Rep%"] / 100
    p["samp_sd"] = 100 * np.sqrt((pd_ + pr_ - (pd_ - pr_) ** 2) / p["SampleSize"])  # SD of (D - R) from sampling alone
    mae = p["Pollster"].map(cfg["TYPICAL_ERROR_BY_POLLSTER"]).astype(float).fillna(p["Scorecard Error"]).fillna(cfg["DEFAULT_TYPICAL_ERROR"])
    p["poll_error_sd"], p["w_accuracy"] = accuracy_parts(p["samp_sd"], mae, cfg)
    if cfg["UNRATED_POLLSTERS"] == "random" and unr.any():  # tables: average over the graded pool
        pool = grade_pool(data, cfg)
        pes, wa = accuracy_parts(p.loc[unr, "samp_sd"].to_numpy()[:, None], pool["mae"][None, :], cfg)
        p.loc[unr, "w_quality"] = pool["quality"].mean()
        p.loc[unr, "w_accuracy"] = wa.mean(axis=1)
        p.loc[unr, "poll_error_sd"] = np.sqrt((pes ** 2).mean(axis=1))
    wcols = [c for c in p.columns if c.startswith("w_")]
    p["weight"] = p[wcols].prod(axis=1)  # final weight = product of all factors
    p["w_base"] = p["weight"] / (p["w_quality"] * p["w_accuracy"])  # everything except the pollster's grade
    return p


def estimate_rv_shift(data, cfg, today):
    """MARKS: average (LV margin - RV margin) over 2026 surveys released both ways, shrunk toward 0. Returns (shift, pairs)."""
    gaps = []
    sp = data["polls"]
    sp = sp[(sp["Date"] <= today) & (sp["Released"].isna() | (sp["Released"] <= today)) & ~sp["Generic"]]
    for _, g in sp.groupby(["State", "Pollster", "Date", "Bigger_Field"]):
        lv, rv = g[g["LVorRV"] == "LV"], g[g["LVorRV"] == "RV"]
        if len(lv) and len(rv):
            gaps.append(float((lv["Dem%"] - lv["Rep%"]).mean() - (rv["Dem%"] - rv["Rep%"]).mean()))
    gb = national_poll_frame(data["generic"], today)
    for _, g in gb.groupby(["Pollster", "Date"]):
        lv, rv = g[g["Population"] == "LV"], g[g["Population"] == "RV"]
        if len(lv) and len(rv):
            gaps.append(float((lv["Dem"] - lv["Rep"]).mean() - (rv["Dem"] - rv["Rep"]).mean()))
    n = len(gaps)
    shift = float(np.clip(sum(gaps) / (n + cfg["RV_SHIFT_PRIOR_PAIRS"]), *cfg["RV_SHIFT_CLIP"])) if n else 0.0
    return shift, n


def rv_shift(cfg):
    """The RV shift in use: the estimate stored by run_model, or the fixed CONFIG number."""
    return cfg["_rv_shift"] if cfg["RV_MARGIN_SHIFT"] == "estimate" else float(cfg["RV_MARGIN_SHIFT"])


def in_cycle_house_effects(p, cfg, prior):
    """MARKS: each pollster's lean this cycle, from polls in the last HOUSE_EFFECT_WINDOW_DAYS. Residual = poll - the same
    state's weighted average WITHOUT that pollster;
    lean = (sum of residuals + k x prior) / (n + k). Re-estimated HOUSE_EFFECT_ITERS times so the comparison polls are
    corrected too. Partisan polls are left out of the estimate (they get PARTISAN_HOUSE_EFFECTS instead).
    Returns per-poll (n, sum of residuals, lean)."""
    recent = (p["age"] <= cfg["HOUSE_EFFECT_WINDOW_DAYS"]).to_numpy()
    x, w = p["adj_margin_raw"].astype(float), p["weight"].astype(float).where(recent, 0.0)
    use = ~p["partisan_poll"] & recent
    k = cfg["HOUSE_EFFECT_K"]
    lean = prior.copy()
    for _ in range(cfg["HOUSE_EFFECT_ITERS"]):
        xc = x - lean
        tot = (w * xc).groupby(p["State"]).transform("sum"); tw = w.groupby(p["State"]).transform("sum")
        own = (w * xc).groupby([p["State"], p["Pollster"]]).transform("sum"); ow = w.groupby([p["State"], p["Pollster"]]).transform("sum")
        others = tw - ow
        res = (x - (tot - own) / others.where(others > 0)).where(use)
        s = res.groupby(p["Pollster"]).agg(["sum", "count"])
        n_i, sum_i = p["Pollster"].map(s["count"]).fillna(0.0), p["Pollster"].map(s["sum"]).fillna(0.0)
        lean = pd.Series(np.where(~p["partisan_poll"], (sum_i + k * prior) / (n_i + k), 0.0), index=p.index)
    return n_i, sum_i, lean


def adjust_margins(p, cfg, data):
    """RV shift, partisan sponsor shift, scorecard lean, optional data-driven house effects."""
    p = p.copy()
    if p.empty:
        return p
    adj = p["margin"].astype(float).copy()
    adj += np.where(p["LVorRV"] == "RV", rv_shift(cfg), 0.0)  # RV skew (MARKS: measured)
    effect = p["Partisan"].map(cfg["PARTISAN_HOUSE_EFFECTS"]).astype(float).fillna(0.0)
    toward_d = p["Partisan"].fillna("").str.startswith("Republican")  # R-sponsored -> shift toward D
    adj += np.where(toward_d, effect, -effect)
    p["partisan_poll"] = p["Partisan"].notna().to_numpy(dtype=bool)
    p["adj_margin_raw"] = adj.copy()  # before the scorecard lean (simulations redraw it for ungraded firms)
    lean = p["Scorecard Lean"].astype(float).copy()
    if cfg["UNRATED_POLLSTERS"] == "random":
        lean[p["Unrated"]] = grade_pool(data, cfg)["lean"].mean()  # average lean of the pool
    p["scorecard_lean"] = np.where(cfg["SCORECARD_HOUSE_EFFECTS"] & ~p["partisan_poll"], lean.fillna(0.0), 0.0)
    if cfg["HOUSE_EFFECT_MODE"] == "in_cycle":  # MARKS: measured this year, shrunk toward the scorecard lean
        p["he_n"], p["he_sum"], p["house_lean"] = in_cycle_house_effects(p, cfg, pd.Series(p["scorecard_lean"], index=p.index))
    else:
        p["he_n"], p["he_sum"], p["house_lean"] = 0.0, 0.0, p["scorecard_lean"]
    adj -= p["house_lean"]  # + lean = leans D -> subtract
    p["house_effect_est"] = 0.0
    if cfg["ESTIMATE_HOUSE_EFFECTS"]:
        w = p["weight"]
        state_avg = (adj * w).groupby(p["State"]).transform("sum") / w.groupby(p["State"]).transform("sum")
        g = (adj - state_avg).groupby(p["Pollster"])
        he = g.sum() / (g.count() + cfg["HOUSE_EFFECT_SHRINKAGE"])  # shrunk mean residual per pollster
        p["house_effect_est"] = p["Pollster"].map(he)
        adj -= p["house_effect_est"]
    p["adj_margin"] = adj
    return p


POLL_AVG_COLS = ["poll_mean", "poll_sd", "n_polls", "n_surveys", "n_eff", "undecided", "last_poll_age"]


def polling_averages(p, cfg):
    rows = {}
    for st, g in p.groupby("State"):
        w = g["weight"].to_numpy()
        mean = np.average(g["adj_margin"], weights=w)
        n_eff = w.sum() ** 2 / (w ** 2).sum()  # Kish effective number of polls
        poll_sd = np.sqrt(np.average(g["poll_error_sd"] ** 2, weights=w))  # typical single-poll SD
        und_sd = np.average(g["undecided_sd"], weights=w)
        sd = np.sqrt(poll_sd ** 2 / n_eff + cfg["SYSTEMATIC_POLL_ERROR_SD"] ** 2 + und_sd ** 2)
        rows[st] = {"poll_mean": mean, "poll_sd": sd, "n_polls": len(g), "n_surveys": g["family"].nunique(),
                    "n_eff": n_eff, "undecided": np.average(g["undecided"], weights=w), "last_poll_age": g["age"].min()}
    return pd.DataFrame.from_dict(rows, orient="index", columns=POLL_AVG_COLS).astype(float)

## Step 5: National environment

Two estimates of the national D margin, blended by `GENERIC_BALLOT_BLEND` (Model 1):

- **Approval:** `env = ENV_SLOPE × −(net approval − APPROVAL_NEUTRAL_NET) + MIDTERM_PENALTY` (sign flips if `PRESIDENT_PARTY = "D"`).
- **Generic ballot:** weighted average margin of recent generic-ballot polls.

**What changed:** the approval polls now come from VoteHub (the spreadsheet is no longer updated), and each signal also gets **Kalshi's polling-average markets** as an always-on extra source (from DadBod). VoteHub's national coverage has been thin since June, and the Kalshi averages summarize many more polls:

$$\text{signal} = \frac{n_{\text{eff}} \cdot \text{VoteHub poll average} + K \cdot \text{Kalshi average}}{n_{\text{eff}} + K}, \qquad K = \texttt{NAT\_KALSHI\_POLL\_EQUIV} \;(5)$$

- **MARKS: polls count as fresh-poll equivalents.** Pray counted VoteHub's polls by their effective number (Kish n_eff), which ignores age: ten generic-ballot polls 25–90 days old still outvoted the live Kalshi average. Here each poll counts 0.5^(age ÷ `NAT_FRESH_HALF_LIFE_DAYS`), capped at the Kish n_eff, so a stale feed cedes weight to the market automatically.
- **MARKS: `GENERIC_BALLOT_BLEND` = 0.75.** The generic ballot is already a congressional margin; approval needs an assumed slope (`ENV_SLOPE`) that nobody has calibrated.
- **RCP and VoteHub approval averages are averaged together first**, then blended, since they track mostly the same polls.
- Kalshi gives **approve %** only. Net = 2 × approve − 100 + U, where **U** = the average share answering neither in VoteHub approval polls over the last 90 days (`NAT_MIX_LOOKBACK_DAYS`), measured each run.
- Kalshi's generic-ballot average is the raw D − R gap; with `GB_MARGIN_BASIS = "two_party"` it's converted: raw × 100 ÷ (D% + R%), using the average D% + R% in recent VoteHub polls.
- **Reading a market:** bucket markets (RCP) → probability-weighted middle of the buckets; weekly "above X%?" markets → X + Φ⁻¹(price) × 0.5.
- **Backtests** skip the Kalshi averages (their history isn't reliably available) and use polls only.

In [ ]:
def national_poll_frame(df, today):
    g = df[df["Date"] <= today].copy()
    if "Released" in g.columns:
        g = g[g["Released"].isna() | (g["Released"] <= today)]  # backtests: only polls already published
    g["Population"] = g["Population"].fillna("").astype(str).str.upper().replace({"ADULTS": "A", "V": "RV"})
    return g


def approval_environment(data, cfg, today):
    """Weighted VoteHub net approval -> (net, n polls, n_eff). Model 1's weighting."""
    n = national_poll_frame(data["approval"], today)
    if n.empty:
        return np.nan, 0, 0.0
    n["Pollster"] = normalize_pollsters(n["Pollster"], cfg["POLLSTER_NAME_FIXES"])
    if cfg["DROP_EXACT_DUPLICATES"]:
        n = n.drop_duplicates(subset=["Date", "Pollster", "Approve", "Disapprove", "SampleSize", "Population"])
    ss = pd.to_numeric(n["SampleSize"], errors="coerce").fillna(cfg["NAT_ZERO_SAMPLE_IMPUTE"]).replace(0, cfg["NAT_ZERO_SAMPLE_IMPUTE"])
    pop = n["Population"].replace({"A": "Adults"})
    net = n["Approve"] - n["Disapprove"] + pop.map(cfg["NAT_NET_APPROVAL_SHIFT"]).fillna(0.0)
    age = (today - n["Date"]).dt.days.clip(lower=0)
    w = 0.5 ** (age / cfg["NAT_HALF_LIFE_DAYS"])
    w *= np.sqrt(ss / cfg["NAT_REFERENCE_SAMPLE"])
    w *= pop.map(cfg["NAT_POPULATION_WEIGHTS"]).fillna(1.0)
    w *= np.where(n["Partisan"].astype(str).str.lower().isin(["true", "1"]), cfg["NAT_PARTISAN_WEIGHT"], 1.0)
    if cfg["NAT_USE_QUALITY"]:
        lo, hi = cfg["QUALITY_SCORE_MIN"], cfg["QUALITY_SCORE_MAX"]
        look = n["Pollster"].map(scorecard_lookup(data["scorecards"]))
        s = pd.Series([cfg["GRADE_TO_RANK"].get(x[0], cfg["UNRANKED_SCORE"]) for x in look], index=n.index).clip(lo, hi)
        w *= cfg["MIN_QUALITY_WEIGHT"] + (1 - cfg["MIN_QUALITY_WEIGHT"]) * (s - lo) / (hi - lo)
    w = w.to_numpy(dtype=float)
    n_fresh = float((0.5 ** (age / cfg["NAT_FRESH_HALF_LIFE_DAYS"])).sum())  # MARKS: fresh-poll equivalents
    return float(np.average(net, weights=w)), len(n), min(float(w.sum() ** 2 / (w ** 2).sum()), n_fresh)


def generic_ballot_poll_average(data, cfg, today):
    """Weighted VoteHub generic-ballot margin -> (margin, n polls, n_eff). Model 1's weighting."""
    g = national_poll_frame(data["generic"], today)
    if g.empty:
        return np.nan, 0, 0.0
    g["Pollster"] = normalize_pollsters(g["Pollster"], cfg["POLLSTER_NAME_FIXES"])
    rank = {pop: i for i, pop in enumerate(cfg["GB_POPULATION_PREFERENCE"])}
    g["pop_rank"] = g["Population"].map(rank).fillna(len(rank))
    g = g.sort_values("pop_rank").drop_duplicates(["Date", "Pollster"], keep="first")  # one version per survey
    recent = g[g["Date"] >= today - pd.Timedelta(days=cfg["GB_LOOKBACK_DAYS"])]
    if len(recent) < cfg["GB_MIN_POLLS"]:
        recent = g.nlargest(cfg["GB_MIN_POLLS"], "Date")
    if recent.empty:
        return np.nan, 0, 0.0
    d, r = recent["Dem"].astype(float), recent["Rep"].astype(float)
    margin = 100 * (d - r) / (d + r) if cfg["GB_MARGIN_BASIS"] == "two_party" else d - r
    if cfg["GB_APPLY_RV_SHIFT"]:
        margin = margin + np.where(recent["Population"] == "RV", rv_shift(cfg), 0.0)
    age = (today - recent["Date"]).dt.days.clip(lower=0)
    n = pd.to_numeric(recent["SampleSize"], errors="coerce").fillna(cfg["NAT_REFERENCE_SAMPLE"]).replace(0, cfg["NAT_REFERENCE_SAMPLE"])
    pop = recent["Population"].replace({"A": "Adults"})
    w = 0.5 ** (age / cfg["GB_HALF_LIFE_DAYS"])
    w *= np.sqrt(n / cfg["NAT_REFERENCE_SAMPLE"])
    w *= pop.map(cfg["NAT_POPULATION_WEIGHTS"]).fillna(1.0)
    w *= np.where(recent["Partisan"].astype(str).str.lower().isin(["true", "1"]), cfg["NAT_PARTISAN_WEIGHT"], 1.0)
    w = w.to_numpy(dtype=float)
    n_fresh = float((0.5 ** (age / cfg["NAT_FRESH_HALF_LIFE_DAYS"])).sum())  # MARKS: fresh-poll equivalents
    return float(np.average(margin, weights=w)), len(recent), min(float(w.sum() ** 2 / (w ** 2).sum()), n_fresh)


def kalshi_implied_value(g):
    """Level of a polling average implied by one Kalshi event's prices."""
    g = g.dropna(subset=["p_mid"])
    if g.empty:
        return np.nan
    fl = pd.to_numeric(g["floor_strike"], errors="coerce")
    cp = pd.to_numeric(g["cap_strike"], errors="coerce")
    if (g["strike_type"] == "greater").all():  # "above X%?" market(s)
        if len(g) == 1:
            p = float(np.clip(g["p_mid"].iloc[0], 0.02, 0.98))
            return float(fl.iloc[0] + norm.ppf(p) * SOURCES["KALSHI_UPDOWN_SD"])
        lad = pd.DataFrame({"x": fl, "p": g["p_mid"].astype(float)}).sort_values("x")  # ladder: where P(above) = 50%
        return float(np.interp(0.5, lad["p"].to_numpy()[::-1], lad["x"].to_numpy()[::-1]))
    strikes = np.sort(pd.concat([fl, cp]).dropna().unique())  # bucket market
    step = float(np.min(np.diff(strikes))) if len(strikes) > 1 else 0.1
    val = np.where(g["strike_type"] == "less", cp - step, np.where(g["strike_type"] == "greater", fl + step, (fl + cp) / 2))
    p = g["p_mid"].astype(float).to_numpy()
    return float(np.sum(p * val) / p.sum())


def national_environment(data, cfg, today):
    backtest = is_backtest(today)
    look = cfg["NAT_MIX_LOOKBACK_DAYS"]
    def recent(df):
        d = national_poll_frame(df, today)
        r = d[d["Date"] >= today - pd.Timedelta(days=look)]
        return r if len(r) >= 10 else d.nlargest(10, "Date")
    ap_mix, gb_mix = recent(data["approval"]), recent(data["generic"])
    U = float((100 - ap_mix["Approve"] - ap_mix["Disapprove"]).clip(lower=0).mean()) if len(ap_mix) else 4.0
    DR = float((gb_mix["Dem"] + gb_mix["Rep"]).mean()) if len(gb_mix) else 90.0

    net_poll, n_ap, ne_ap = approval_environment(data, cfg, today)
    gb_poll, n_gb, ne_gb = generic_ballot_poll_average(data, cfg, today) if cfg["USE_GENERIC_BALLOT"] else (np.nan, 0, 0.0)

    kn = data["kalshi_national"]
    parts = {} if backtest or kn is None or kn.empty else {
        (sig, src): kalshi_implied_value(g) for (sig, src), g in kn.groupby(["signal", "source"])}
    ap_k = [v for (sig, _), v in parts.items() if sig == "approval" and not np.isnan(v)]
    gb_k = [v for (sig, _), v in parts.items() if sig == "generic_ballot" and not np.isnan(v)]
    approve_k = float(np.mean(ap_k)) if ap_k else np.nan  # RCP + VoteHub combined FIRST
    net_k = 2 * approve_k - 100 + U if ap_k else np.nan
    gb_k_val = np.nan
    if gb_k:
        gb_k_val = float(np.mean(gb_k)) * (100 / DR if cfg["GB_MARGIN_BASIS"] == "two_party" else 1.0)

    K = cfg["NAT_KALSHI_POLL_EQUIV"]
    def blend(poll, n_eff, kal):
        pts = [(w, v) for w, v in [(n_eff, poll), (K, kal)] if w > 0 and not np.isnan(v)]
        return sum(w * v for w, v in pts) / sum(w for w, _ in pts) if pts else np.nan

    net = blend(net_poll, ne_ap, net_k)
    gb = blend(gb_poll, ne_gb, gb_k_val) if cfg["USE_GENERIC_BALLOT"] else np.nan
    sign = 1 if cfg["PRESIDENT_PARTY"] == "R" else -1
    env_appr = sign * (cfg["ENV_SLOPE"] * -(net - cfg["APPROVAL_NEUTRAL_NET"]) + cfg["MIDTERM_PENALTY"]) if not np.isnan(net) else np.nan
    env_gb = gb + cfg["GB_SHIFT"] if not np.isnan(gb) else np.nan
    if cfg["ENV_OVERRIDE"] is not None:
        env = cfg["ENV_OVERRIDE"]
    elif np.isnan(env_appr) and np.isnan(env_gb):
        print("WARNING: no approval or generic-ballot data as of this date; environment set to 0")
        env = 0.0
    elif np.isnan(env_gb):
        env = env_appr
    elif np.isnan(env_appr):
        env = env_gb
    else:
        b = cfg["GENERIC_BALLOT_BLEND"]
        env = b * env_gb + (1 - b) * env_appr
    table = pd.DataFrame([
        {"signal": "generic ballot (D-R)", "VoteHub polls": gb_poll, "n polls": n_gb, "effective n (fresh)": ne_gb,
         "Kalshi average": gb_k_val, "Kalshi counts as": K if not np.isnan(gb_k_val) else 0, "combined": gb},
        {"signal": "net approval", "VoteHub polls": net_poll, "n polls": n_ap, "effective n (fresh)": ne_ap,
         "Kalshi average": net_k, "Kalshi counts as": K if not np.isnan(net_k) else 0, "combined": net},
    ]).set_index("signal")
    return {"net_approval": net, "env_approval": env_appr, "env_generic": env_gb, "n_generic": n_gb, "env": env,
            "table": table, "U": U, "DR": DR, "kalshi_approve_pct": approve_k, "kalshi_parts": parts, "backtest": backtest}

## Step 6: Fundamentals prior + Bayesian update (with Kalshi)

**Prior** (Model 1's, plus MARKS' candidate term):

$$\text{prior} = \text{PVI\_TO\_MARGIN} \times \text{PVI} + \text{elasticity} \times \text{env} \pm \text{incumbency} + w_c \times \text{candidate effect} + \text{manual adjustment}$$

**Candidate effects (MARKS).** A lean-based prior treats every nominee as a generic Democrat or Republican. That was the biggest reason Pray disagreed with Kalshi: Maine's prior was D+13.5 against Susan Collins, who won by 8.6 in 2020 while Biden carried Maine by 9. `candidate_history.csv` lists each competitive-race nominee's most recent **top-of-ticket statewide** race (Senate, governor, at-large House) with its source. For each one:

$$\text{over-performance} = \text{their margin} - \underbrace{(\text{state pres. margin} - \text{national pres. margin} + \text{national House margin})}_{\text{a generic candidate that year}}$$

Both nominees' over-performances are added (in D − R points), capped at ±`CANDIDATE_EFFECT_CLIP`, and multiplied by `CANDIDATE_EFFECT_WEIGHT` (w_c). Past Senate races say about **0.3** of a candidate's past over-performance carries into the next race (calibration). Down-ballot offices (attorney general, lieutenant governor) aren't used: they mostly measure the ticket, not the candidate. Rows are skipped automatically if the person isn't the current nominee.

**Kalshi's implied margin (MARKS changes).** Three fixes to Pray's conversion:
1. **A floor on the conversion SD.** Pray used the model's own uncertainty to turn a price into a margin, which is circular in well-polled races (a more confident model made Kalshi look more confident). Now margin = Φ⁻¹(P') × max(`KALSHI_CONVERT_SD`, the race's prior+polls SD). The floor (4.7 pts) is the historical final-weeks error of a polled Senate race; unpolled races keep their wider SD, because the market faces the same lack of information.
2. **Longshot correction.** Prediction markets overprice underdogs. P' = logistic(`KALSHI_LONGSHOT_SLOPE` × logit P), which moves 5% to about 3% and 68% to about 70%.
3. **Only informative prices.** Outside `KALSHI_USE_RANGE` (6–94%) a price sits at the 1–5¢ floor (tick size, fees, longshot bias) and only says "safe"; turning it into a margin dragged safe races toward the middle (Wyoming R+39 → R+35 in Pray). Those races now ignore Kalshi.

With `KALSHI_SD` = 8 (Pray: 17) the market is a real third source: about 10–20% of a well-polled race and up to about half of an unpolled one.

**Bayesian update with three sources** (unchanged): each counts by its precision π = 1 ÷ SD²; with `CORRELATED_ERRORS` the range is widened for sources that miss together:

$$\text{var} = \frac{1}{\Pi} + \frac{2\rho}{\Pi^2}\left(\sqrt{\pi_F\pi_P} + \sqrt{\pi_F\pi_K} + \sqrt{\pi_P\pi_K}\right), \qquad \Pi = \pi_F + \pi_P + \pi_K$$

Then extra uncertainty for the days left before the election is added (`DAILY_DRIFT_SD`; zero on election day).


In [ ]:
def reliability(n_eff, cfg):
    """Extra poll-precision factor r (1 = Model 1's pure Bayesian behaviour)."""
    mode = cfg["POLL_RELIABILITY"]
    n_eff = np.nan_to_num(np.asarray(n_eff, dtype=float))
    if mode == "bayesian" or float(mode) == 0:
        return np.ones_like(n_eff)
    k = float(mode)
    return np.where(n_eff > 0, n_eff / (n_eff + k), 0.0)


def posterior(prior_mean, prior_sd, poll_mean, poll_sd, n_eff, k_margin, cfg, drift):
    """Precision-weighted combination of prior, polls and Kalshi. Works on (races,) or (sims, races) arrays.
    Returns mean, sd (incl. drift), and each source's share of the mean."""
    prior_mean, prior_sd, poll_mean, poll_sd, k_margin = (np.asarray(x, dtype=float) for x in
                                                          (prior_mean, prior_sd, poll_mean, poll_sd, k_margin))
    has_p, has_k = ~np.isnan(poll_mean), ~np.isnan(k_margin) & kalshi_on(cfg)
    pi_f = 1.0 / np.asarray(prior_sd, dtype=float) ** 2 * np.ones_like(poll_mean)
    with np.errstate(divide="ignore", invalid="ignore"):
        pi_p = np.where(has_p, cfg["POLL_PRECISION_MULT"] * reliability(n_eff, cfg) / poll_sd ** 2, 0.0)
    pi_k = np.where(has_k, 1.0 / (cfg["KALSHI_SD"] or 1.0) ** 2, 0.0)
    P = pi_f + pi_p + pi_k
    wf, wp, wk = pi_f / P, pi_p / P, pi_k / P
    mean = wf * prior_mean + wp * np.nan_to_num(poll_mean) + wk * np.nan_to_num(k_margin)
    var = 1.0 / P
    if cfg["CORRELATED_ERRORS"]:  # same weights; errors of the three sources correlated with rho
        var = var + 2 * cfg["SOURCE_ERROR_CORR"] / P ** 2 * (np.sqrt(pi_f * pi_p) + np.sqrt(pi_f * pi_k) + np.sqrt(pi_p * pi_k))
    return mean, np.sqrt(var + drift ** 2), wf, wp, wk


def drift_sd(cfg, today):
    days_left = max((pd.Timestamp(cfg["ELECTION_DAY"]) - today).days, 0)
    return cfg["DAILY_DRIFT_SD"] * (np.sqrt(days_left) if cfg["DRIFT_SHAPE"] == "sqrt" else days_left)


def kalshi_as_of(data, today):
    """P(opposition wins) per race (+ CONTROL = P(D control)): live prices for today, else each market's last daily
    close on/before that date (backtests)."""
    k = data["kalshi_races"]
    if not is_backtest(today):
        out = pd.DataFrame({"p_opp": 1 - k[k["side"] == "R"].set_index("State")["p_mid"].astype(float)})
        ctrl = k.loc[k["State"] == "CONTROL", "p_mid"]
        out.loc["CONTROL", "p_opp"] = float(ctrl.iloc[0]) if len(ctrl) else np.nan
        return out
    if "kalshi_history" not in data:
        data["kalshi_history"] = with_cache("kalshi_history_cache.csv", fetch_kalshi_history, [])
    h = data["kalshi_history"]
    cutoff = (today + pd.Timedelta(days=1)).tz_localize("America/New_York").timestamp()  # end of that day, Eastern
    h = h[h["end_period_ts"] <= cutoff].sort_values("end_period_ts").groupby("market_ticker").last()
    bid, ask, last = (pd.to_numeric(h[c], errors="coerce") for c in ("yes_bid", "yes_ask", "last"))
    price = pd.Series(np.where(bid.notna() & ask.notna() & (bid > 0) & (ask >= bid), (bid + ask) / 2, last), index=h.index)
    out = {}
    for st in list(CONFIG["SEATS_UP"]) + ["CONTROL"]:
        side = "D" if st == "CONTROL" else "R"
        t = k.loc[(k["State"] == st) & (k["side"] == side), "market_ticker"]
        p = price.get(t.iloc[0], np.nan) if len(t) else np.nan
        out[st] = p if st == "CONTROL" else 1 - p
    return pd.DataFrame({"p_opp": pd.Series(out)})


def load_candidate_history(cfg):
    """MARKS: candidate_history.csv -> one row per nominee with a usable past race."""
    path = Path(cfg["CANDIDATE_FILE"])
    if not path.exists():
        return pd.DataFrame()
    h = pd.read_csv(path, comment="#")
    h = h[h["use"].astype(str).str.lower().isin(["yes", "y", "true", "1"])].copy()
    h["baseline"] = h["state_pres_margin"] - h["national_pres_margin"] + h["national_house_margin"]
    h["overperformance"] = h["race_margin"] - h["baseline"]  # D - R points; + = better for the D/independent side
    return h


def candidate_effects(cfg, nominees, history):
    """MARKS: {state: pts added to the prior} and a table of what was used."""
    if not cfg["CANDIDATE_EFFECTS"] or history.empty:
        return pd.Series(0.0, index=sorted(cfg["SEATS_UP"])), pd.DataFrame()
    year = pd.Timestamp(cfg["ELECTION_DAY"]).year
    rows = []
    for _, r in history.iterrows():
        st = r["state"]
        if st not in nominees:
            continue
        rep, opp = nominees[st]
        current = {surname(rep)} | {surname(n) for n in opp}
        is_nominee = surname(r["candidate"]) in current
        recent = year - int(r["year"]) <= cfg["CANDIDATE_MAX_AGE_YEARS"]
        ov = float(np.clip(r["overperformance"], -cfg["CANDIDATE_EFFECT_CLIP"], cfg["CANDIDATE_EFFECT_CLIP"]))
        rows.append({"State": st, "candidate": r["candidate"], "race": f"{int(r['year'])} {r['office']}",
                     "their margin": r["race_margin"], "generic baseline": r["baseline"], "over-performance (D-R)": ov,
                     "used": bool(is_nominee and recent),
                     "why not": "" if is_nominee and recent else ("not the 2026 nominee" if not is_nominee else "too old")})
    t = pd.DataFrame(rows)
    used = t[t["used"]] if len(t) else t
    eff = (used.groupby("State")["over-performance (D-R)"].sum() * cfg["CANDIDATE_EFFECT_WEIGHT"]) if len(used) else pd.Series(dtype=float)
    return eff.reindex(sorted(cfg["SEATS_UP"])).fillna(0.0), t


def kalshi_to_margin(p_opp, sd_model, cfg):
    """MARKS: longshot-corrected price -> margin with a fixed SD; NaN outside KALSHI_USE_RANGE (price only says 'safe')."""
    p = pd.Series(p_opp, dtype=float)
    lo, hi = cfg["KALSHI_USE_RANGE"]
    pc = p.clip(*SOURCES["KALSHI_PROB_CLIP"])
    p_adj = 1 / (1 + np.exp(-cfg["KALSHI_LONGSHOT_SLOPE"] * np.log(pc / (1 - pc))))
    sd = sd_model if cfg["KALSHI_CONVERT_SD"] == "model" else np.maximum(float(cfg["KALSHI_CONVERT_SD"]), sd_model)
    m = pd.Series(norm.ppf(p_adj) * sd, index=p.index)
    return m.where(p.between(lo, hi)), p_adj


def build_race_table(data, poll_avg, env, cfg, today):
    races = pd.DataFrame(index=sorted(cfg["SEATS_UP"]))
    races.index.name = "State"
    races["incumbent_party"] = pd.Series(cfg["SEATS_UP"])
    races["status"] = pd.Series(cfg["INCUMBENT_STATUS"]).reindex(races.index).fillna(cfg["DEFAULT_INCUMBENT_STATUS"])
    races["independent"] = races.index.isin(list(cfg["INDEPENDENT_CAUCUS_PROB_D"]))
    races = races.join(data["leans"][["PartisanLean", "Region"]])
    missing = races.index[races["PartisanLean"].isna()].tolist()
    assert not missing, f"races with no partisan lean: {missing}"
    elasticity = pd.Series(cfg["ENV_ELASTICITY"], dtype=float).reindex(races.index).fillna(cfg["ENV_ELASTICITY_DEFAULT"])
    adv = races["status"].map({"running": cfg["INCUMBENCY_ADVANTAGE"],
                               "appointed": cfg["APPOINTED_INCUMBENT_ADVANTAGE"], "open": 0.0}).fillna(0.0)
    races["incumbency_pts"] = np.where(races["incumbent_party"] == "D", adv, -adv)  # + helps D, - helps R
    races["manual_adj"] = pd.Series(cfg["STATE_PRIOR_ADJUSTMENTS"], dtype=float).reindex(races.index).fillna(0.0)
    cand, _ = candidate_effects(cfg, NOMINEES, CANDIDATE_HISTORY)
    races["candidate_pts"] = cand.reindex(races.index).fillna(0.0)  # MARKS
    races["prior_mean"] = (cfg["PVI_TO_MARGIN"] * races["PartisanLean"] + elasticity * env
                           + races["incumbency_pts"] + races["candidate_pts"] + races["manual_adj"])
    races["prior_sd"] = cfg["PRIOR_SD"] * np.where(races["independent"], cfg["INDEPENDENT_PRIOR_SD_MULT"], 1.0)

    races = races.join(poll_avg)  # NaN for states with no polls
    for c in ["n_polls", "n_surveys"]:
        races[c] = races[c].fillna(0).astype(int)
    races["n_eff"] = races["n_eff"].fillna(0.0)
    races["reliability"] = np.where(races["poll_mean"].notna(), reliability(races["n_eff"], cfg), np.nan)
    drift = drift_sd(cfg, today)
    no_k = np.full(len(races), np.nan)
    pf = posterior(races["prior_mean"], races["prior_sd"], races["poll_mean"], races["poll_sd"], races["n_eff"], no_k, cfg, drift)
    races["pf_mean"], races["pf_sd"] = pf[0], pf[1]  # prior + polls only (Model 1's estimate); pf_sd scales Kalshi
    ktable = kalshi_as_of(data, today)
    races["kalshi_p"] = ktable["p_opp"].reindex(races.index).astype(float)
    races["kalshi_margin"], races["kalshi_p_adj"] = kalshi_to_margin(races["kalshi_p"], races["pf_sd"], cfg)  # MARKS
    m, s, wf, wp, wk = posterior(races["prior_mean"], races["prior_sd"], races["poll_mean"], races["poll_sd"],
                                 races["n_eff"], races["kalshi_margin"], cfg, drift)
    races["post_mean"], races["post_sd"] = m, s
    races["share_prior"], races["share_polls"], races["share_kalshi"] = wf, wp, wk
    races["poll_share_of_estimate"] = wp  # Model 1's column name
    return races, ktable


CANDIDATE_HISTORY = load_candidate_history(CONFIG)
_, CANDIDATE_TABLE = candidate_effects(CONFIG, NOMINEES, CANDIDATE_HISTORY)


## Step 7: Simulations

**MARKS: errors built from components in points** (`ERROR_MODEL = "components"`). In every simulated election, race *i* in region *g* comes out at

$$\text{margin}_i = \text{estimate}_i + \underbrace{N}_{\text{national}} + \underbrace{R_g}_{\text{regional}} + \underbrace{L \cdot f_i}_{\text{lean factor}} + \underbrace{e_i}_{\text{state}}$$

| Piece | Size | Shared by | From |
|---|---|---|---|
| N | `NATIONAL_ERROR_SD` (history ≈ 2.4 pts), Student-t with `NATIONAL_T_DF` | every race | how much each year's Senate polls missed on average, 1998–2022 |
| R_g | `REGIONAL_ERROR_SD` (≈ 1.6) | races in the same Census region | the extra miss shared within a region in a year |
| L·f_i | `LEAN_FACTOR_ERROR_SD` × the state's lean ÷ `LEAN_FACTOR_SCALE` (capped at 2) | red states one way, blue states the other | the 2016/2020-style miss that grew with Trump's vote |
| e_i | whatever is left of the race's own SD (at least `MIN_STATE_ERROR_SD`), t with `STATE_T_DF` | that race only | — |

The state piece is sized so each race keeps **its own total uncertainty** (post_sd), so race probabilities are unchanged in spirit; what changes is how races move *together*. Compared with Pray (one correlation matrix scaled by each race's SD, and one fat-tail draw multiplying every race at once):
- a national miss is the **same number of points everywhere**, instead of larger in unpolled safe states;
- **only the national miss has fat tails**, so a bad polling year can still hit every race, but simulations where Democrats lose Massachusetts (Pray's 35-seat tail) disappear.

`ERROR_MODEL = "legacy"` restores Pray's simulation exactly. Independent draws (each race's own probability, as a check) use the same pieces drawn separately per race.

**Ungraded pollsters get a random grade in every simulation** (unchanged from Pray): each ungraded firm gets a randomly drawn VoteHub-graded firm's grade, past error and lean, and the state's poll average, poll SD and Bayesian estimate are recomputed for that simulation.

**Deciding seat (from Rylyn's model).** In each simulation, races are ordered from best to worst for Democrats; the race that delivers seat number 51 is that simulation's deciding seat. The share of simulations in which each race decides control shows where the majority is really won or lost.


In [ ]:
def unit_shocks(rng, n_sims, k, cfg, L=None):
    """Unit-variance shocks: normal or Student-t, independent or correlated via Cholesky factor L."""
    z = rng.standard_normal((n_sims, k))
    if L is not None:
        z = z @ L.T
    if cfg["USE_T_DIST"]:
        df = cfg["T_DF"]
        shape = (n_sims, 1) if (L is not None and cfg["SHARED_TAIL_SCALE"]) else (n_sims, k)
        g = rng.chisquare(df, size=shape) / df
        z = z / np.sqrt(g) * np.sqrt((df - 2) / df)  # t draws rescaled to variance 1
    return z


def nearest_correlation(R):
    """Clip negative eigenvalues so overrides can't break the Cholesky step."""
    vals, vecs = np.linalg.eigh(R)
    R2 = vecs @ np.diag(np.clip(vals, 1e-6, None)) @ vecs.T
    d = np.sqrt(np.diag(R2))
    return R2 / np.outer(d, d)


def correlation_matrix(races, cfg):
    lean = races["PartisanLean"].to_numpy(dtype=float)
    region = races["Region"].to_numpy(dtype=object)
    same_region = (region[:, None] == region[None, :]).astype(float)
    lean_sim = np.exp(-np.abs(lean[:, None] - lean[None, :]) / cfg["LEAN_SCALE"])
    R = cfg["CORR_NATIONAL"] + cfg["CORR_REGION"] * same_region + cfg["CORR_LEAN"] * lean_sim
    idx = {s: i for i, s in enumerate(races.index)}
    for (a, b), rho in cfg["STATE_PAIR_CORR_OVERRIDES"].items():
        if a in idx and b in idx:
            R[idx[a], idx[b]] = R[idx[b], idx[a]] = rho
    np.fill_diagonal(R, 1.0)
    return nearest_correlation(R)


def unrated_draws(polls, races, cfg, data, n_sims, rng, today, chunk=2000):
    """(means, sds), each (n_sims, races): the Bayesian estimate recomputed in every simulation after giving each
    ungraded pollster a random graded pollster's grade, error and lean. None if not needed."""
    if cfg["UNRATED_POLLSTERS"] != "random" or polls.empty or not polls["Unrated"].any():
        return None
    pool = grade_pool(data, cfg)
    unr = polls["Unrated"].to_numpy(dtype=bool)
    codes, uniq = pd.factorize(polls.loc[unr, "Pollster"])  # one draw per POLLSTER per simulation
    nonpart = ~polls["partisan_poll"].to_numpy(dtype=bool)[unr]
    ind = (polls["State"].to_numpy()[:, None] == races.index.to_numpy()[None, :]).astype(float)  # poll x race
    base_w, base_a, base_e = (polls[c].to_numpy(dtype=float) for c in ("weight", "adj_margin", "poll_error_sd"))
    und = polls["undecided_sd"].to_numpy(dtype=float)
    a_raw_u = (polls["adj_margin_raw"] - polls["house_effect_est"]).to_numpy(dtype=float)[unr]
    he_n_u, he_sum_u = (polls[c].to_numpy(dtype=float)[unr] for c in ("he_n", "he_sum"))
    w_base_u, samp_u = polls.loc[unr, "w_base"].to_numpy(dtype=float), polls.loc[unr, "samp_sd"].to_numpy(dtype=float)
    drift = drift_sd(cfg, today)
    means, sds = np.empty((n_sims, len(races))), np.empty((n_sims, len(races)))
    for s0 in range(0, n_sims, chunk):
        n = min(chunk, n_sims - s0)
        draw = rng.integers(0, len(pool["quality"]), size=(n, len(uniq)))[:, codes]  # (n, ungraded polls)
        pes_u, wa_u = accuracy_parts(samp_u[None, :], pool["mae"][draw], cfg)
        W, A, E = np.tile(base_w, (n, 1)), np.tile(base_a, (n, 1)), np.tile(base_e, (n, 1))
        W[:, unr] = w_base_u * pool["quality"][draw] * wa_u
        E[:, unr] = pes_u
        if cfg["SCORECARD_HOUSE_EFFECTS"]:
            lean = pool["lean"][draw]
            if cfg["HOUSE_EFFECT_MODE"] == "in_cycle":  # MARKS: the random draw is the starting value of the measured lean
                k_ = cfg["HOUSE_EFFECT_K"]
                lean = (he_sum_u[None, :] + k_ * lean) / (he_n_u[None, :] + k_)
            A[:, unr] = np.where(nonpart, a_raw_u - lean, A[:, unr])
        sw = W @ ind
        with np.errstate(invalid="ignore", divide="ignore"):
            pm = ((W * A) @ ind) / sw
            n_eff = sw ** 2 / ((W ** 2) @ ind)
            single = ((W * E ** 2) @ ind) / sw
            und_sd = ((W * und) @ ind) / sw
            psd = np.sqrt(single / n_eff + cfg["SYSTEMATIC_POLL_ERROR_SD"] ** 2 + und_sd ** 2)
        m, s, *_ = posterior(races["prior_mean"].to_numpy(), races["prior_sd"].to_numpy(), pm, psd, n_eff,
                             np.broadcast_to(races["kalshi_margin"].to_numpy(), pm.shape), cfg, drift)
        means[s0:s0 + n], sds[s0:s0 + n] = m, s
    return means, sds


def t_unit(rng, df, shape):
    """Unit-variance Student-t draws (normal if df is None)."""
    z = rng.standard_normal(shape)
    if df is None:
        return z
    return z / np.sqrt(rng.chisquare(df, size=shape) / df) * np.sqrt((df - 2) / df)


def component_errors(races, cfg, rng, n_sims, sd, independent=False):
    """MARKS: (n_sims, races) simulated errors in points = national + regional + lean factor + state.
    sd: each race's total SD, shape (races,) or (n_sims, races). independent=True draws the shared pieces separately
    for every race (for the race-by-race check)."""
    k = len(races)
    sN, sR, sL = cfg["NATIONAL_ERROR_SD"], cfg["REGIONAL_ERROR_SD"], cfg["LEAN_FACTOR_ERROR_SD"]
    f = np.clip(races["PartisanLean"].to_numpy(dtype=float) / cfg["LEAN_FACTOR_SCALE"], -2, 2)
    region_codes, regions = pd.factorize(races["Region"])
    if independent:
        nat = sN * t_unit(rng, cfg["NATIONAL_T_DF"], (n_sims, k))
        reg = sR * rng.standard_normal((n_sims, k))
        lean = sL * rng.standard_normal((n_sims, k)) * f
    else:
        nat = sN * t_unit(rng, cfg["NATIONAL_T_DF"], (n_sims, 1))
        reg = (sR * rng.standard_normal((n_sims, len(regions))))[:, region_codes]
        lean = sL * rng.standard_normal((n_sims, 1)) * f[None, :]
    shared_var = sN ** 2 + sR ** 2 + (sL * f) ** 2
    s_state = np.sqrt(np.maximum(np.asarray(sd, dtype=float) ** 2 - shared_var, cfg["MIN_STATE_ERROR_SD"] ** 2))
    return nat + reg + lean + s_state * t_unit(rng, cfg["STATE_T_DF"], (n_sims, k))


def state_simulations(races, cfg, rng, n_sims, draws=None):
    mu, sd = draws if draws is not None else (races["post_mean"].to_numpy(), races["post_sd"].to_numpy())
    if cfg["ERROR_MODEL"] == "components":
        races["p_dem_state_sim"] = ((mu + component_errors(races, cfg, rng, n_sims, sd, independent=True)) > 0).mean(axis=0)
        return races
    z = unit_shocks(rng, n_sims, len(races), cfg)  # independent shocks (legacy)
    races["p_dem_state_sim"] = ((mu + sd * z) > 0).mean(axis=0)
    return races


def national_simulations(races, cfg, rng, n_sims, draws=None):
    k = len(races)
    mu, sd = draws if draws is not None else (races["post_mean"].to_numpy(), races["post_sd"].to_numpy())
    if cfg["ERROR_MODEL"] == "components":  # MARKS
        margins = mu + component_errors(races, cfg, rng, n_sims, sd)
    else:  # legacy: Pray / Model 1
        L = np.linalg.cholesky(correlation_matrix(races, cfg))
        margins = mu + sd * unit_shocks(rng, n_sims, k, cfg, L=L)
    opp_wins = margins > 0  # D or credible independent wins
    caucus_d = np.ones_like(opp_wins, dtype=bool)
    for st, prob in cfg["INDEPENDENT_CAUCUS_PROB_D"].items():
        if st in races.index:
            caucus_d[:, races.index.get_loc(st)] = rng.random(n_sims) < prob  # weighted coin per simulation
    d_seats = cfg["HOLDOVER_D"] + (opp_wins & caucus_d).sum(axis=1)
    for _, prob in cfg["HOLDOVER_DEFECTION_PROB"].items():
        d_seats = d_seats - (rng.random(n_sims) < prob)
    races["p_dem_national_sim"] = opp_wins.mean(axis=0)
    races["p_dem"] = races["p_dem_national_sim"]  # short name used by Model Testing and the standard export
    return races, pd.DataFrame(margins, columns=races.index), d_seats


def deciding_seats(sims, cfg):
    """MARKS (from Rylyn): share of simulations in which each race delivers Democrats' control-th seat."""
    need = control_threshold(cfg) - cfg["HOLDOVER_D"]  # races Democrats must win
    m = sims.to_numpy()
    order = np.argsort(-m, axis=1)  # best race for D first
    pivot = order[:, need - 1]
    share = pd.Series(np.bincount(pivot, minlength=m.shape[1]) / len(m), index=sims.columns)
    return share.sort_values(ascending=False)


## Step 8: Run the full model

`run_model()` chains every step. It only reads `DATA` and the `cfg` you pass in, so Model Testing can call it with modified copies of `CONFIG`. A second run with Kalshi switched off (`KALSHI_SD = None`) shows how much Kalshi moved the answer.

In [ ]:
def run_model(data, cfg, n_sims=None, verbose=True):
    validate_config(cfg)
    n_sims = int(n_sims or cfg["N_SIMS"])
    rng = np.random.default_rng(cfg["SEED"])
    today = resolve_today(cfg)
    cfg = dict(cfg)
    cfg["_rv_shift"], n_pairs = estimate_rv_shift(data, cfg, today) if cfg["RV_MARGIN_SHIFT"] == "estimate" else (cfg["RV_MARGIN_SHIFT"], 0)
    polls = clean_polls(data["polls"], cfg, today, verbose)
    polls = apply_version_mode(polls, cfg, verbose)
    polls = compute_margins(polls, cfg)
    polls = weight_polls(polls, cfg, today, data)
    polls = adjust_margins(polls, cfg, data)
    poll_avg = polling_averages(polls, cfg) if len(polls) else pd.DataFrame(columns=POLL_AVG_COLS, dtype=float)
    env = national_environment(data, cfg, today)
    races, ktable = build_race_table(data, poll_avg, env["env"], cfg, today)
    draws = unrated_draws(polls, races, cfg, data, n_sims, rng, today)
    races = state_simulations(races, cfg, rng, n_sims, draws)
    races, sims, d_seats = national_simulations(races, cfg, rng, n_sims, draws)
    need = control_threshold(cfg)
    summary = {"p_d_51": float((d_seats >= need).mean()), "exp_d_seats": float(d_seats.mean()),
               "median_d_seats": float(np.median(d_seats)), "p10": float(np.percentile(d_seats, 10)),
               "p90": float(np.percentile(d_seats, 90)), "env": float(env["env"])}
    summary["p_d_control"], summary["mean_d_seats"] = summary["p_d_51"], summary["exp_d_seats"]  # Model 1's names
    summary["rv_shift"], summary["rv_pairs"] = float(cfg["_rv_shift"]), int(n_pairs)
    if verbose:
        f = lambda x: "n/a" if x is None or np.isnan(x) else f"{x:+.1f}"
        print(f"As of {today:%Y-%m-%d}: net approval {f(env['net_approval'])} -> approval env D{f(env['env_approval'])}; "
              f"generic-ballot env D{f(env['env_generic'])}; blended env D{env['env']:+.1f}"
              + (" (backtest: national = polls only)" if env["backtest"] else ""))
        if draws is not None:
            u = polls[polls["Unrated"]]
            print(f"Ungraded pollsters: {u['Pollster'].nunique()} ({len(u)} of {len(polls)} polls, "
                  f"{u['weight'].sum() / polls['weight'].sum():.0%} of poll weight) get a random VoteHub grade in each "
                  f"of the {n_sims:,} simulations")
        print(f"RV shift: {cfg['_rv_shift']:+.2f} pts" + (f" (measured from {n_pairs} LV/RV survey pairs)" if n_pairs else " (fixed)"))
        print(f"D seats: median {summary['median_d_seats']:.0f}, 80% interval {summary['p10']:.0f}-{summary['p90']:.0f} | "
              f"P(D control, >= {need}) = {summary['p_d_51']:.1%}")
    return {"polls": polls, "poll_avg": poll_avg, "env": env, "races": races, "sims": sims, "d_seats": d_seats,
            "summary": summary, "today": today, "kalshi": ktable, "draws": draws, "deciding": deciding_seats(sims, cfg)}


RES = run_model(DATA, CONFIG)
RES_NO_KALSHI = run_model(DATA, {**CONFIG, "KALSHI_SD": None}, verbose=False)  # same model, Kalshi switched off
display(RES["env"]["table"].round(2))

## Step 9: Results

### The two headline numbers, next to the other models
The other models' rows come from their latest saved standard output (`Output/standard/summary.json`), so re-run a model (or `run_all.py`) to refresh its row.

In [ ]:
s, s0 = RES["summary"], RES_NO_KALSHI["summary"]
kt = RES["kalshi"]
k_ctrl = float(kt.loc["CONTROL", "p_opp"]) if "CONTROL" in kt.index else np.nan
kp = kt["p_opp"].drop("CONTROL", errors="ignore")
k_seats = CONFIG["HOLDOVER_D"] + sum(kp.get(st, np.nan) * CONFIG["INDEPENDENT_CAUCUS_PROB_D"].get(st, 1.0) for st in CONFIG["SEATS_UP"])
print(f"P(Democrats hold at least {control_threshold(CONFIG)} seats): {s['p_d_51']:.1%}")
print(f"Predicted Democratic seats (mean):   {s['exp_d_seats']:.1f}   (median {s['median_d_seats']:.0f}, 80% range {s['p10']:.0f}-{s['p90']:.0f})")


def other_models():
    asof = f"{RES['today']:%Y-%m-%d}"
    rows = [{"model": "Marks_Final — this run", "P(D >= 51)": s["p_d_51"], "D seats (mean)": s["exp_d_seats"], "as of": asof},
            {"model": "Marks_Final, Kalshi off", "P(D >= 51)": s0["p_d_51"], "D seats (mean)": s0["exp_d_seats"], "as of": asof},
            {"model": "Kalshi alone (control market / sum of race odds)", "P(D >= 51)": k_ctrl, "D seats (mean)": k_seats, "as of": asof}]
    for f in sorted((REPO_ROOT / "Models").glob("*/**/Output/standard/summary.json")):
        if MODEL_DIR in f.parents:
            continue
        try:
            h = json.loads(f.read_text())
            name = h.get("model", "?") + (f" ({h['variant']})" if h.get("variant") else "")
            rows.append({"model": name, "P(D >= 51)": h["p_d_51"], "D seats (mean)": h["exp_d_seats"], "as of": h.get("as_of")})
        except Exception:
            pass
    return pd.DataFrame(rows).set_index("model")


COMPARE = other_models()
try:
    display(COMPARE.style.format({"P(D >= 51)": "{:.1%}", "D seats (mean)": "{:.1f}"}))
except (AttributeError, ImportError):  # jinja2 missing: plain table
    display(COMPARE.round(3))

### Under the hood: one state's poll weights
Change `INSPECT_STATE` to look at another race. Each `w_` column is one weighting factor; `weight` is their product. For ungraded pollsters (`Unrated = True`) with random grades, `w_quality` and `w_accuracy` show the pool *average*.

In [ ]:
INSPECT_STATE = "Maine"  # display only, not a model setting
p = RES["polls"]
if len(p):
    wcols = [c for c in p.columns if c.startswith("w_") and c != "w_base"]
    show_cols = ["Pollster", "Grade", "Unrated", "Date", "LVorRV", "Dem%", "Rep%", "undecided", "margin", "scorecard_lean", "adj_margin"]
    display(p.loc[p["State"] == INSPECT_STATE].assign(Date=lambda d: d["Date"].dt.date)[show_cols + wcols + ["weight"]]
            .sort_values("weight", ascending=False).round(3))

### Race-by-race table
- **prior / polls / kalshi_margin:** each source's margin. **post_mean:** the combined (posterior) margin; **post_sd:** its uncertainty.
- **share_*:** how much of the final margin came from each source (they add to 100%). `poll_share_of_estimate` = `share_polls` (Model 1's name).
- **p_dem_state_sim / p_dem_national_sim:** P(D or independent wins) from the independent and correlated simulations (they should be close).

In [ ]:
cols = ["incumbent_party", "status", "PartisanLean", "Region", "n_polls", "n_eff", "undecided", "incumbency_pts", "candidate_pts", "prior_mean", "poll_mean",
        "kalshi_p", "kalshi_p_adj", "kalshi_margin", "share_prior", "share_polls", "share_kalshi", "post_mean", "post_sd",
        "p_dem_state_sim", "p_dem_national_sim"]
race_table = RES["races"][cols].sort_values("post_mean")
race_table.round(2)

### Win probabilities and ratings
Ratings use `RATING_TOSSUP` / `RATING_LEAN` / `RATING_LIKELY` on the favorite's win probability.

In [ ]:
def margin_label(m):
    return "-" if pd.isna(m) else (f"D+{m:.1f}" if m >= 0 else f"R+{-m:.1f}")


def race_rating(p_dem, state, cfg):
    opp = "Ind" if state in cfg["INDEPENDENT_CAUCUS_PROB_D"] else "D"
    fav, top = (opp, p_dem) if p_dem >= 0.5 else ("R", 1 - p_dem)
    if top < cfg["RATING_TOSSUP"]:
        return "Toss-up"
    return ("Lean" if top < cfg["RATING_LEAN"] else "Likely" if top < cfg["RATING_LIKELY"] else "Safe") + f" {fav}"


r = RES["races"]
win_table = pd.DataFrame({
    "margin": r["post_mean"].map(margin_label), "Model 1-style (no Kalshi)": r["pf_mean"].map(margin_label),
    "P(D/Ind win)": r["p_dem"], "Kalshi P": r["kalshi_p"], "model - Kalshi": r["p_dem"] - r["kalshi_p"],
    "rating": [race_rating(pv, st, CONFIG) for st, pv in r["p_dem"].items()],
}).sort_values("P(D/Ind win)", ascending=False)
try:
    display(win_table.style.format({"P(D/Ind win)": "{:.0%}", "Kalshi P": "{:.0%}", "model - Kalshi": "{:+.0%}"}))
except (AttributeError, ImportError):
    display(win_table.round(2))

### Candidate effects (MARKS)
Each nominee's most recent top-of-ticket statewide race against what a generic candidate of their party would have done that year (from `candidate_history.csv`). The prior moves by `CANDIDATE_EFFECT_WEIGHT` × the sum for both nominees. Positive = better for the Democrat/independent.


In [ ]:
if len(CANDIDATE_TABLE):
    ct = CANDIDATE_TABLE.copy()
    ct["pts added to prior"] = np.where(ct["used"], ct["over-performance (D-R)"] * CONFIG["CANDIDATE_EFFECT_WEIGHT"], 0.0)
    try:
        display(ct.style.format({"their margin": "{:+.1f}", "generic baseline": "{:+.1f}", "over-performance (D-R)": "{:+.1f}",
                                 "pts added to prior": "{:+.1f}"}))
    except (AttributeError, ImportError):
        display(ct.round(1))
else:
    print("No candidate history used (CANDIDATE_EFFECTS off or candidate_history.csv missing).")


### Pollster house effects this cycle (MARKS)
Each pollster's measured lean this year: its polls compared with other pollsters' polls of the same states, shrunk toward its scorecard lean (`HOUSE_EFFECT_K`). **Negative = more Republican than other pollsters**; that amount is added back to its polls. Ungraded firms show their in-table value (starting from 0); in the simulations their starting value is redrawn every time.


In [ ]:
_p = RES["polls"][~RES["polls"]["partisan_poll"]]
if CONFIG["HOUSE_EFFECT_MODE"] == "in_cycle" and len(_p):
    he_tab = (_p.groupby("Pollster")
              .agg(polls=("he_n", "first"), graded=("Unrated", lambda x: not x.iloc[0]),
                   avg_residual=("he_sum", lambda x: x.iloc[0]), scorecard_lean=("scorecard_lean", "first"), lean_used=("house_lean", "first"))
              .assign(avg_residual=lambda t: t["avg_residual"] / t["polls"].where(t["polls"] > 0)))
    he_tab = he_tab[he_tab["polls"] >= 3].sort_values("lean_used")
    try:
        display(he_tab.style.format({"polls": "{:.0f}", "avg_residual": "{:+.1f}", "scorecard_lean": "{:+.1f}", "lean_used": "{:+.1f}"}))
    except (AttributeError, ImportError):
        display(he_tab.round(1))
else:
    print("HOUSE_EFFECT_MODE = 'scorecard': fixed VoteHub leans only.")


### Deciding seat (from Rylyn's model)
In each simulation, races are ranked from best to worst for Democrats; the race that delivers seat 51 decides control in that simulation. A race with a high share is where the majority is won or lost.


In [ ]:
dec = pd.DataFrame({"deciding-seat share": RES["deciding"], "P(D/Ind win)": RES["races"]["p_dem"].reindex(RES["deciding"].index),
                    "margin": RES["races"]["post_mean"].reindex(RES["deciding"].index).map(margin_label)})
dec = dec[dec["deciding-seat share"] >= 0.005]
try:
    display(dec.style.format({"deciding-seat share": "{:.1%}", "P(D/Ind win)": "{:.0%}"}))
except (AttributeError, ImportError):
    display(dec.round(3))


### Correlation in action
How winning one state changes the odds in a related state (the "if Kansas swings R, Missouri probably did too" idea).

In [ ]:
def conditional_demo(sims, a, b):
    pa, pb = sims[a] > 0, sims[b] > 0
    print(f"P(D wins {b}) = {pb.mean():.2f} | if D wins {a}: {pb[pa].mean():.2f} | if D loses {a}: {pb[~pa].mean():.2f}")


for a, b in [("Iowa", "Kansas"), ("Maine", "New Hampshire"), ("Texas", "Ohio"), ("Michigan", "Minnesota")]:
    conditional_demo(RES["sims"], a, b)

### Model vs Kalshi
Points off the diagonal are races where the model and the market disagree. Above the line = the model is more bullish on Democrats than Kalshi.

In [ ]:
def plot_vs_kalshi(r, path=None):
    fig, ax = plt.subplots(figsize=(8, 8))
    ax.plot([0, 1], [0, 1], color="#BBBBBB", lw=1, zorder=1)
    ax.scatter(r["kalshi_p"], r["p_dem"], s=40, color="#1A6FB0", zorder=3)
    for st, row in r.iterrows():
        if 0.1 < row["kalshi_p"] < 0.95 or abs(row["p_dem"] - row["kalshi_p"]) > 0.1:
            ax.annotate(st, (row["kalshi_p"], row["p_dem"]), xytext=(5, -3), textcoords="offset points", fontsize=8, color="#3C3C3C")
    ax.set_xlabel("Kalshi P(D / Ind win)")
    ax.set_ylabel("Model P(D / Ind win)")
    ax.set_xlim(-0.02, 1.02); ax.set_ylim(-0.02, 1.02)
    ax.grid(color="#EEEEEE", zorder=0)
    ax.spines[["right", "top"]].set_visible(False)
    ax.set_title("Marks_Final vs Kalshi, race by race", loc="left", fontweight="bold")
    fig.tight_layout()
    if path:
        fig.savefig(path, dpi=200, bbox_inches="tight")
    plt.show()


plot_vs_kalshi(RES["races"], CONFIG["OUT_DIR"] / "model_vs_kalshi.png")

### Seat distribution
Blue bars = Democratic control (incl. caucusing independents); red = Republican control. The triangle marks the average.

In [ ]:
def plot_seat_distribution(d_seats, cfg, today, path=None):
    DEM_BLUE, REP_RED, INK = "#1A6FB0", "#C8312B", "#3C3C3C"
    need = control_threshold(cfg)
    seats, counts = np.unique(d_seats, return_counts=True)
    pct = 100 * counts / len(d_seats)
    p_dem = (d_seats >= need).mean()
    fig, ax = plt.subplots(figsize=(11, 6))
    ax.bar(seats, pct, width=0.85, color=np.where(seats >= need, DEM_BLUE, REP_RED), zorder=3)
    ax.axvline(need - 0.5, color=INK, lw=1.5, zorder=2)
    ax.plot([d_seats.mean()], [0], marker="^", ms=13, color=INK, clip_on=False, zorder=5, ls="none",
            label=f"Mean {d_seats.mean():.1f} D seats")
    top = pct.max() * 1.15
    ax.text(need - 0.8, top, f"R control\n{1 - p_dem:.0%}", ha="right", va="top", color=INK, fontsize=14, fontweight="bold")
    ax.text(need - 0.2, top, f"D {need}+\n{p_dem:.0%}", ha="left", va="top", color=INK, fontsize=14, fontweight="bold")
    ax.set_ylim(0, top * 1.05)
    ax.set_xlim(np.percentile(d_seats, 0.1) - 1.5, np.percentile(d_seats, 99.9) + 1.5)
    ax.set_xlabel("Democratic seats (incl. caucusing independents)")
    ax.set_ylabel("Share of simulations (%)")
    ax.grid(axis="y", color="#E3E3E3", zorder=0)
    ax.spines[["right", "top"]].set_visible(False)
    ax.legend(loc="upper right", frameon=False)
    ax.set_title(f"Marks_Final: {len(d_seats):,} simulated elections (as of {today:%b %d, %Y})", loc="left", fontweight="bold")
    fig.tight_layout()
    if path:
        fig.savefig(path, dpi=200, bbox_inches="tight")
    plt.show()


plot_seat_distribution(RES["d_seats"], CONFIG, RES["today"], CONFIG["OUT_DIR"] / "seat_distribution.png")

### Save outputs
A JSON snapshot of `CONFIG` and the headline numbers is saved next to the results, so every output can be traced back to the assumptions that produced it.

In [ ]:
out = CONFIG["OUT_DIR"]
_json = lambda x: x.item() if hasattr(x, "item") else str(x)
race_table.to_csv(out / "race_probabilities.csv")
win_table.to_csv(out / "state_win_probabilities.csv")
RES["polls"].to_csv(out / "polls_used.csv", index=False)
RES["env"]["table"].to_csv(out / "national_signals.csv")
pd.Series(RES["d_seats"]).value_counts().sort_index().rename("n_sims").to_csv(out / "seat_distribution.csv", index_label="dem_seats")
with open(out / "config_snapshot.json", "w") as f:
    json.dump({k: v for k, v in CONFIG.items()}, f, indent=2, default=_json)
headline = {"as_of": f"{RES['today']:%Y-%m-%d}", "run_at": pd.Timestamp.now().strftime("%Y-%m-%d %H:%M"), **RES["summary"],
            "kalshi_off": RES_NO_KALSHI["summary"], "kalshi_p_control": k_ctrl, "kalshi_expected_seats": k_seats,
            "national": {"generic_ballot": RES["env"]["env_generic"], "net_approval": RES["env"]["net_approval"],
                         "env": RES["env"]["env"], "undecided_share_U": RES["env"]["U"]},
            "sources": SOURCE_STATUS}
with open(out / "headline.json", "w") as f:
    json.dump(headline, f, indent=2, default=_json)
print(f"Saved race_probabilities.csv, state_win_probabilities.csv, polls_used.csv, national_signals.csv, "
      f"seat_distribution.csv/.png, model_vs_kalshi.png, config_snapshot.json, headline.json to {out}")

### Standard export for `run_all.py`
Every model in the repo writes the same three files to `Output/standard/` so `run_all.py` can compare them without knowing how each model works: `summary.json` (headline numbers + key settings), `races.csv` (one row per race) and `seats.csv` (seat distribution). See the repo README.

In [ ]:
STD = CONFIG["OUT_DIR"] / "standard"
STD.mkdir(parents=True, exist_ok=True)
_r = RES["races"]
pd.DataFrame({"state": _r.index, "p_dem": _r["p_dem"].values, "margin": _r["post_mean"].values, "sd": _r["post_sd"].values,
              "kalshi_p": _r["kalshi_p"].values, "n_polls": _r["n_polls"].values}).to_csv(STD / "races.csv", index=False)
_sh = pd.Series(RES["d_seats"]).value_counts(normalize=True).sort_index()
pd.DataFrame({"dem_seats": _sh.index, "share": _sh.values}).to_csv(STD / "seats.csv", index=False)
_c = CONFIG
_ks = f"1/{_c['KALSHI_SD']:g}^2" if kalshi_on(_c) else "0 (comparison only)"
STANDARD = {
    "model": "Marks_Final", "variant": "", "run_at": pd.Timestamp.now().strftime("%Y-%m-%d %H:%M"),
    "as_of": f"{RES['today']:%Y-%m-%d}", **{k: RES["summary"][k] for k in ("p_d_51", "exp_d_seats", "median_d_seats", "p10", "p90")},
    "n_sims": int(_c["N_SIMS"]), "kalshi_p_control": k_ctrl, "kalshi_expected_seats": k_seats, "sources": SOURCE_STATUS,
    "key_settings": {
        "Poll source": "VoteHub API",
        "Poll age": (f"all polls, half-life {_c['RECENCY_HALF_LIFE_DAYS']} days" if _c["RECENCY_MODE"] == "half_life"
                     else f"last {_c['WINDOW_DAYS']} days, weight 1.0 -> {_c['WINDOW_OLDEST_WEIGHT']}"),
        "Polls : fundamentals : Kalshi": (f"Bayesian precision (poll floor {_c['SYSTEMATIC_POLL_ERROR_SD']:.1f}, prior SD {_c['PRIOR_SD']:.1f}); "
                                          f"Kalshi precision {_ks}, longshot-corrected, conversion SD >= {_c['KALSHI_CONVERT_SD']}"),
        "Poll reliability (thin races)": ("automatic (Bayesian precision)" if _c["POLL_RELIABILITY"] == "bayesian"
                                          else f"Bayesian x n_eff/(n_eff+{_c['POLL_RELIABILITY']})"),
        "Fundamentals": (f"PVI x {_c['PVI_TO_MARGIN']:g} + national environment (generic-ballot blend {_c['GENERIC_BALLOT_BLEND']:g})"
                         + (f" + candidate effects x {_c['CANDIDATE_EFFECT_WEIGHT']:.2f}" if _c["CANDIDATE_EFFECTS"] else "")),
        "Incumbency": f"{_c['INCUMBENCY_ADVANTAGE']:.1f} pts (appointed {_c['APPOINTED_INCUMBENT_ADVANTAGE']:g})",
        "LV : RV : A": " : ".join(f"{v:g}" for v in _c["POPULATION_WEIGHTS"].values())
                       + f"; RV shift {RES['summary']['rv_shift']:+.1f} ({'measured' if _c['RV_MARGIN_SHIFT'] == 'estimate' else 'fixed'})",
        "Flooding penalty": f"1/k^{_c['FLOOD_EXPONENT']}",
        "House effects": ((f"in-cycle, shrunk to scorecard (k {_c['HOUSE_EFFECT_K']:g})" if _c["HOUSE_EFFECT_MODE"] == "in_cycle"
                           else "on (scorecard)") if _c["SCORECARD_HOUSE_EFFECTS"] else "partisan sponsors only")
                         + (" + estimated" if _c["ESTIMATE_HOUSE_EFFECTS"] else ""),
        "Ungraded pollsters": _c["UNRATED_POLLSTERS"] if _c["UNRATED_POLLSTERS"] == "random" else f"rank {_c['UNRANKED_SCORE']} (fixed)",
        "National signals": f"VoteHub polls + Kalshi averages (= {_c['NAT_KALSHI_POLL_EQUIV']} polls)",
        "Kalshi races": "Bayesian input + comparison (found automatically)" if kalshi_on(_c) else "comparison only",
        "Source errors": (f"correlated (rho {_c['SOURCE_ERROR_CORR']})" if _c["CORRELATED_ERRORS"] else "independent (Bayesian)")
                         + (f"; sims: national {_c['NATIONAL_ERROR_SD']:.1f} + region {_c['REGIONAL_ERROR_SD']:.1f} + state, in pts"
                            if _c["ERROR_MODEL"] == "components" else "; sims: legacy correlated t"),
        "Calibration": (f"calibration.json ({CALIBRATION.get('created', '?')})" if CALIBRATION else "none (CONFIG as written)"),
    },
}
with open(STD / "summary.json", "w") as f:
    json.dump(STANDARD, f, indent=2, default=_json)
print(f"Standard export -> {STD}")

---
**Next:** open **`Model Testing.ipynb`** in this folder. It re-runs this notebook, then checks the data, flags thin or stale inputs, stress-tests the settings (sensitivity sweep), and charts how the forecast has moved week by week.